In [ ]:
import numpy as np
import scipy as sp
from typing import Callable, Union
import os
os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=true intra_op_parallelism_threads=0"
import jax
import jax.numpy as jnp
jax.config.update('jax_enable_x64', True)
import pickle
import logging
import time
import sys
import matplotlib.pyplot as plt
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
lgcg_path = os.path.abspath(os.path.join('../nlgcg'))
if lgcg_path not in sys.path:
    sys.path.append(lgcg_path)
from lib.measure import Measure
from lib.ssn import SSN
from lib.particle_descent import ParticleDescent
from lib.adaptive_refinement import AdaptiveRefinement
from nlgcg import NLGCG

# Heat Equation

## Generate Data and Define Functions

In [ ]:
logging.getLogger().setLevel(logging.INFO) # Supress logging

In [ ]:
# Omega = np.array([[0,1], [0,1]])
# alpha = 1e-1
# observation_resolution = 4
# std_factor = 0.1
# true_sources = np.array([[0.28, 0.71], [0.51,0.27], [0.71,0.53]])
# true_weights = np.array([1,-0.7, 0.8])
# true_measure = Measure(support=true_sources, coefficients=true_weights)

In [ ]:
Omega = np.array([[0,1], [0,1]])
alpha = 2e-1
observation_resolution = 10
std_factor = 0.02
np.random.seed(49)
true_sources = np.random.rand(20, 2)*0.9
true_weights = np.random.rand(20) * 2 - 1
true_measure = Measure(support=true_sources, coefficients=true_weights)

In [ ]:
observations = (np.array(np.meshgrid(
                    *(
                        np.linspace(bound[0], bound[1], observation_resolution+2)
                        for bound in Omega
                    ))
            ).reshape(len(Omega), -1).T)
observations = np.array([obs for obs in observations if all(obs!=0) and all(obs!=1)])

In [ ]:
@jax.jit
def kernel(omega: np.ndarray):
    outer_factor = np.sqrt(std_factor*np.pi)**Omega.shape[0]
    norms = -jnp.square(jnp.linalg.norm(omega-observations, axis=1))/std_factor # (len(x),)
    exponentiated = jnp.exp(norms)/outer_factor # (len(x),)
    return exponentiated

grad_kernel = jax.jit(jax.jacobian(kernel))
hess_kernel = jax.jit(jax.hessian(kernel))
_ = grad_kernel(true_sources[0])
_ = hess_kernel(true_sources[0])

kernel = jax.vmap(kernel)
grad_kernel = jax.vmap(grad_kernel)
hess_kernel = jax.vmap(hess_kernel)

In [ ]:
target = true_measure.duality_pairing(kernel)

@jax.jit
def g(w: np.ndarray) -> float:
    return alpha * jnp.linalg.norm(w, ord=1)
# grad_g = jax.jit(jax.grad(g))
# _ = grad_g(jnp.ones(1))

@jax.jit
def f(y: np.ndarray) -> float:
    return 0.5 * jnp.sum((y - target)**2)
f_grad = jax.jit(jax.grad(f))
_ = f_grad(jnp.zeros(target.shape[0]))

j = lambda u: f(u.duality_pairing(kernel)) + g(u.coefficients)

In [ ]:
def p(u):
    inner = -f_grad(u.duality_pairing(kernel, len(target)))
    return lambda omega: kernel(omega) @ inner

def grad_P(u):
    inner = -f_grad(u.duality_pairing(kernel, len(target)))
    return lambda omega: np.tensordot(grad_kernel(omega), inner, axes=([1,0]))

def hess_P(u):
    inner = -f_grad(u.duality_pairing(kernel, len(target)))
    return lambda omega: np.tensordot(hess_kernel(omega), inner, axes=([1,0]))

In [ ]:
@jax.jit
def j_N(input: np.ndarray):
    input = input.reshape(-1, Omega.shape[0]+1)
    weights = input[:,0]
    omega = input[:,1:]
    return f(kernel(omega).T@weights) + g(weights)

grad_j_N = jax.jit(jax.grad(j_N))
hess_j_N = jax.jit(jax.hessian(j_N))
_ = grad_j_N(np.ones((2, Omega.shape[0]+1)).flatten())
_ = hess_j_N(np.ones((2, Omega.shape[0]+1)).flatten())

## Experiments

In [ ]:
exp = NLGCG(target=target, 
           kernel=kernel, 
           g=g, 
           j=j,
           j_N=j_N,
           p=p,
           grad_P=grad_P,
           hess_P=hess_P,
           grad_j_N=grad_j_N,
           hess_j_N=hess_j_N,
           alpha=alpha,
           Omega=Omega,
           global_search_resolution=4,
           dual_variable_goodness=0.3,
           armijo_constant=0.1
           )

In [ ]:
u, times, supports, inner_loop, lgcg_lazy, lgcg_total, objective_values, dropped_tot, epsilons = exp.nlgcg(tol=1e-12, max_radius=0.1, mode="stochastic")

In [ ]:
np.array(times)

In [ ]:
print(u)

In [ ]:
hesses = exp.hess_P(u)(u.support)
grads = exp.grad_P(u)(u.support)

In [ ]:
np.linalg.eigvals(hesses[5])

In [ ]:
u.to_matrix().shape

In [ ]:
hess = hess_j_N(u.to_matrix().flatten())
np.linalg.eigvals(hess)

## Plots

In [ ]:
p_u = p(u)
a = np.arange(0,1,0.01)
x, y = np.meshgrid(a,a)
points = np.array(list(zip(x.flatten(), y.flatten())))
vals = np.abs(p_u(points)).reshape((100,100))

plt.contourf(x, y, vals, levels=100);
for omega in u.support:
    plt.plot(omega[0], omega[1], "o", c="black", markersize=5);
plt.colorbar();

In [ ]:
p_u = p(u)
P = lambda x: np.abs(p_u(x))
a = np.arange(0,1,0.01)
B, D = np.meshgrid(a,a)
points = np.array(list(zip(B.flatten(), D.flatten())))
vals = P(points).reshape((100,100))

plt.contourf(B, D, vals, levels=100);
plt.colorbar();
plt.plot(Omega[0][0]-1, Omega[0][1]-1, "P", c="black", markersize=8, label="True sources");
plt.plot(Omega[0][0]-1, Omega[0][1]-1, "o", c="black", label="Optimal support");
plt.xlim(Omega[0][0], Omega[0][1]);
plt.ylim(Omega[1][0], Omega[1][1]);
for i, x in enumerate(true_sources):
    if true_weights[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(true_weights[i]) * 12 + 4
    plt.plot([x[0]], [x[1]], "P", alpha=0.5, c=color, markersize=size);
for i, x in enumerate(u.support):
    if u.coefficients[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(u.coefficients[i]) * 10 + 2
    plt.plot([x[0]], [x[1]], "o", c=color, markersize=size);
plt.legend();

In [ ]:
# Plot the measured heat distribution
def heat(x):
    # Input is 2D array of shape (number of points, Omega dimension)
    if len(x.shape) == 1:
        x = x.reshape(1, -1) 
    weighted_heat = np.zeros(x.shape[0]) # (len(x),)
    outer_factor = np.sqrt(std_factor*np.pi)**Omega.shape[0]
    for point, weight in zip(true_sources, true_weights):
        diff = point-x # (len(x), Omega.shape[0])
        norms = -np.square(np.linalg.norm(diff, axis=1))/std_factor # (len(x),)
        exponentiated = np.exp(norms) # (len(x),)
        weighted_heat += exponentiated * weight # (len(x),)
    result = weighted_heat/outer_factor # shape=(len(x),)
    return result

a = np.arange(0,1,0.01)
B, D = np.meshgrid(a,a)
x = np.array(list(zip(B.flatten(), D.flatten())))
true_vals = heat(x).reshape((100,100))

plt.contourf(B, D, true_vals, levels=100);
plt.colorbar();
plt.xlim(Omega[0][0], Omega[0][1]);
plt.ylim(Omega[1][0], Omega[1][1]);
for i, x in enumerate(true_sources):
    if true_weights[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(true_weights[i]) * 12 + 4
    plt.plot([x[0]], [x[1]], "P", c=color, markersize=size);

In [ ]:
def predicted_heat(x):
    # Input is 2D array of shape (number of points, Omega dimension)
    if len(x.shape) == 1:
        x = x.reshape(1, -1) 
    weighted_heat = np.zeros(x.shape[0]) # (len(x),)
    outer_factor = np.sqrt(std_factor*np.pi)**Omega.shape[0]
    for point, weight in zip(u.support, u.coefficients):
        diff = point-x # (len(x), Omega.shape[0])
        norms = -np.square(np.linalg.norm(diff, axis=1))/std_factor # (len(x),)
        exponentiated = np.exp(norms) # (len(x),)
        weighted_heat += exponentiated * weight # (len(x),)
    result = weighted_heat/outer_factor # shape=(len(x),)
    return result

a = np.arange(0,1,0.01)
B, D = np.meshgrid(a,a)
x = np.array(list(zip(B.flatten(), D.flatten())))
pred_vals = predicted_heat(x).reshape((100,100))
error = true_vals - pred_vals

print(f"L2 error: {np.linalg.norm(error)/np.sqrt(len(x)):.3E}, Linf error: {np.max(np.abs(error)):.3E}")

plt.contourf(B, D, np.abs(error), levels=100);
plt.colorbar();
plt.plot(Omega[0][0]-1, Omega[0][1]-1, "P", c="black", markersize=8, label="True sources");
plt.plot(Omega[0][0]-1, Omega[0][1]-1, "o", c="black", label="Optimal support");
plt.xlim(Omega[0][0], Omega[0][1]);
plt.ylim(Omega[1][0], Omega[1][1]);
for i, x in enumerate(true_sources):
    if true_weights[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(true_weights[i]) * 12 + 4
    plt.plot([x[0]], [x[1]], "P", alpha=0.5, c=color, markersize=size);
for i, x in enumerate(u.support):
    if u.coefficients[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(u.coefficients[i]) * 10 + 2
    plt.plot([x[0]], [x[1]], "o", c=color, markersize=size);
plt.legend();

# Signal Processing

## Generate Data and Define Functions

In [ ]:
logging.getLogger().setLevel(logging.INFO) # Supress logging

In [ ]:
observation_resolution = 120
Omega = np.array([[0,observation_resolution//2]])
alpha = 1e-1
true_sources = np.array([[3.125], [7], [np.sqrt(179)]])
true_weights = np.array([-1, 0.7, 0.5])
true_measure = Measure(support=true_sources, coefficients=true_weights)

In [ ]:
max_radius = (Omega[0][1]-Omega[0][0])/100

In [ ]:
# alpha = 1e-0
# source_number = 20
# observation_resolution = 100
# Omega = np.array([[0,observation_resolution//2]])
# np.random.seed(49)
# true_sources = np.array(np.random.rand(source_number)*50)
# true_weights = np.random.rand(source_number) * 2 - 1
# true_measure = Measure(support=true_sources, coefficients=true_weights)

In [ ]:
observations = np.arange(0,1,1/observation_resolution)

In [ ]:
# @jax.jit
# def kernel(omega: np.ndarray):
#     return jnp.sin(2*np.pi*omega*observations).flatten()

# grad_kernel = jax.jit(jax.jacobian(kernel))
# hess_kernel = jax.jit(jax.hessian(kernel))
# _ = grad_kernel(true_sources[0])
# _ = hess_kernel(true_sources[0])

# kernel = jax.vmap(kernel)
# grad_kernel = jax.vmap(grad_kernel)
# hess_kernel = jax.vmap(hess_kernel)

In [ ]:
@jax.jit
def singleton_kernel(omega: float):
    return jnp.sin(2*np.pi*omega*observations)

grad_kernel = jax.jit(jax.jacobian(singleton_kernel))
hess_kernel = jax.jit(jax.hessian(singleton_kernel))
_ = grad_kernel(np.ones(len(Omega)))
_ = hess_kernel(np.ones(len(Omega)))

kernel = jax.vmap(singleton_kernel)
grad_kernel = jax.vmap(grad_kernel)
hess_kernel = jax.vmap(hess_kernel)

In [ ]:
target = true_measure.duality_pairing(kernel)

In [ ]:
@jax.jit
def g(w: np.ndarray) -> float:
    return alpha * jnp.linalg.norm(w, ord=1)
grad_g = jax.jit(jax.grad(g))
_ = grad_g(jnp.ones(1))

@jax.jit
def f(y: np.ndarray) -> float:
    return 0.5 * jnp.sum((y - target)**2)
grad_f = jax.jit(jax.grad(f))
hess_f = jax.jit(jax.hessian(f))
_ = grad_f(jnp.zeros(target.shape[0]))
_ = hess_f(jnp.zeros(target.shape[0]))

j = lambda u, c: f(u.duality_pairing(kernel)+c*np.ones(target.shape)) + g(u.coefficients)

In [ ]:
# def p(u):
#     inner = -f_grad(u.duality_pairing(kernel, len(target)))
#     return lambda omega: kernel(omega) @ inner

# def grad_P(u):
#     inner = -f_grad(u.duality_pairing(kernel, len(target)))
#     return lambda omega: np.tensordot(grad_kernel(omega), inner, axes=([1,0]))

# def hess_P(u):
#     inner = -f_grad(u.duality_pairing(kernel, len(target)))
#     return lambda omega: np.tensordot(hess_kernel(omega), inner, axes=([1,0]))

In [ ]:
@jax.jit
def p_raw(parameters: np.ndarray, c: float, omega: np.ndarray):
    coefficients = parameters[:,0]
    support = parameters[:,1:]
    Ku = jnp.tensordot(kernel(support), coefficients, axes=([0], [0]))
    constant_term = c*jnp.ones(len(target))
    return singleton_kernel(omega) @ -grad_f(Ku+constant_term)

grad_p_raw = jax.jit(jax.grad(p_raw, argnums=2))
hess_p_raw = jax.jit(jax.hessian(p_raw, argnums=2))

p_raw = jax.jit(jax.vmap(p_raw, in_axes=(None, None, 0), out_axes=0))
grad_p_raw = jax.jit(jax.vmap(grad_p_raw, in_axes=(None, None, 0), out_axes=0))
hess_p_raw = jax.jit(jax.vmap(hess_p_raw, in_axes=(None, None, 0), out_axes=0))

p = lambda u, c: lambda omega: p_raw(u.to_matrix(len(Omega)), c, omega)
grad_p = lambda u, c: lambda omega: grad_p_raw(u.to_matrix(len(Omega)), c, omega)
hess_p = lambda u, c: lambda omega: hess_p_raw(u.to_matrix(len(Omega)), c, omega)

In [ ]:
# @jax.jit
# def j_N(input: np.ndarray):
#     input = input.reshape(-1, Omega.shape[0]+1)
#     weights = input[:,0]
#     omega = input[:,1:]
#     return f(kernel(omega).T@weights) + g(weights)

# grad_j_N = jax.jit(jax.grad(j_N))
# hess_j_N = jax.jit(jax.hessian(j_N))
# _ = grad_j_N(np.ones((2, Omega.shape[0]+1)).flatten())
# _ = hess_j_N(np.ones((2, Omega.shape[0]+1)).flatten())

In [ ]:
# Parameterized versions of f and j
@jax.jit
def f_N(raw_input: np.ndarray) -> float:
    constant = raw_input[-1]
    input = raw_input[:-1].reshape(-1, Omega.shape[0]+1)
    weights = input[:,0]
    omega = input[:,1:]
    return f(kernel(omega).T@weights+constant*jnp.ones(target.shape))

@jax.jit
def j_N(raw_input: np.ndarray) -> float:
    input = raw_input[:-1].reshape(-1, Omega.shape[0]+1)
    weights = input[:,0]
    return f_N(raw_input) + g(weights)

grad_f_N = jax.jit(jax.grad(f_N))
hess_f_N = jax.jit(jax.hessian(f_N))
grad_j_N = jax.jit(jax.grad(j_N))
hess_j_N = jax.jit(jax.hessian(j_N))

In [ ]:
# Define functions where the derivatives are taken wrt regularized (weights) and non-regularized(support + constant) parameters

@jax.jit
def f_N_(weights: np.ndarray, support_constant: np.ndarray) -> float:
    constant = support_constant[-1]
    omega = support_constant[:-1].reshape(-1, Omega.shape[0])
    return f(kernel(omega).T@weights+constant*jnp.ones(target.shape))

@jax.jit
def j_N_(weights: np.ndarray, support_constant: np.ndarray) -> float:
    return f_N_(weights, support_constant) + g(weights)

grad_f_N_reg = jax.jit(jax.grad(f_N_, argnums=0))
hess_f_N_reg = jax.jit(jax.hessian(f_N_, argnums=0))
grad_j_N_reg = jax.jit(jax.grad(j_N_, argnums=0))
hess_j_N_reg = jax.jit(jax.hessian(j_N_, argnums=0))

grad_f_N_non_reg = jax.jit(jax.grad(f_N_, argnums=1))
hess_f_N_non_reg = jax.jit(jax.hessian(f_N_, argnums=1))
grad_j_N_non_reg = jax.jit(jax.grad(j_N_, argnums=1))
hess_j_N_non_reg = jax.jit(jax.hessian(j_N_, argnums=1))

## Experiments

### NLGCG

In [ ]:
exp = NLGCG(target=target, 
           kernel=kernel, 
           g=g,
           f=f,
           f_N=f_N,
           grad_f=grad_f,
           hess_f=hess_f,
           grad_f_N=grad_f_N,
           hess_f_N=hess_f_N,
           grad_f_N_non_reg=grad_f_N_non_reg,
           j=j,
           j_N=j_N,
           j_N_=j_N_,
           p=p,
           grad_p=grad_p,
           hess_p=hess_p,
           grad_j_N=grad_j_N,
           hess_j_N=hess_j_N,
           alpha=alpha,
           Omega=Omega,
           global_search_resolution=5,
           dual_variable_goodness=0.3,
           constant_dim=len(target),
           kernel_dim=len(target),
           newton_tolerance=2e-2
           )

In [ ]:
u, c, times, supports, inner_loop, lgcg_lazy, lgcg_total, objective_values, dropped_tot, epsilons = exp.solve(tol=5e-14, max_radius=max_radius, temperature=0.1)

In [ ]:
np.array(times)

In [ ]:
dropped_tot

In [ ]:
print(u)

### Particle Gradient Descent

In [ ]:
a_parameter=0.001
exp = ParticleDescent(
        m=200,
        j=j,
        p=p,
        grad_p=grad_p,
        Omega=Omega,
        a_parameter=a_parameter,
        b_parameter=a_parameter/2,
        kernel=kernel,
        constant_dim=len(target),
        kernel_dim=len(target),
        alpha=alpha,
        target=target,
        g=g,
        f=f,
        grad_f=grad_f,
        hess_f=hess_f,
        ssn_steps=100,
    )

In [ ]:
u, c, objective_values, supports, times, success = exp.solve(max_iters=int(1e6), mode="uniform")

In [ ]:
objective_values[-1]

### Adaptive Refinement

In [ ]:
exp = AdaptiveRefinement(
        observations=observations,
        j=j,
        p=p,
        grad_p=grad_p,
        hess_p=hess_p,
        Omega=Omega,
        kernel=kernel,
        constant_dim=len(target),
        kernel_dim=len(target),
        alpha=alpha,
        target=target,
        g=g,
        f=f,
        grad_f=grad_f,
        hess_f=hess_f,
        ssn_steps=1000,
    )

In [ ]:
cells_dict, vertices_dict, vertices, u, objective_values, times, actives, supports = exp.solve(max_iters=200)

### Full

In [ ]:
exp_nlgcg = NLGCG(target=target, 
           kernel=kernel, 
           g=g,
           f=f,
           f_N=f_N,
           grad_f=grad_f,
           hess_f=hess_f,
           grad_f_N=grad_f_N,
           hess_f_N=hess_f_N,
           grad_f_N_non_reg=grad_f_N_non_reg,
           j=j,
           j_N=j_N,
           j_N_=j_N_,
           p=p,
           grad_p=grad_p,
           hess_p=hess_p,
           grad_j_N=grad_j_N,
           hess_j_N=hess_j_N,
           alpha=alpha,
           Omega=Omega,
           global_search_resolution=10,
           dual_variable_goodness=0.3,
           constant_dim=len(target),
           kernel_dim=len(target),
           newton_tolerance=2e-2
           )

In [ ]:
exp_particle = ParticleDescent(
        m=250,
        j=j,
        p=p,
        grad_p=grad_p,
        Omega=Omega,
        a_parameter=0.001,
        b_parameter=0.001/2,
        kernel=kernel,
        constant_dim=len(target),
        kernel_dim=len(target),
        alpha=alpha,
        target=target,
        g=g,
        f=f,
        grad_f=grad_f,
        hess_f=hess_f,
        ssn_steps=100,
    )

In [ ]:
exp_adaptive = AdaptiveRefinement(
        observations=observations,
        j=j,
        p=p,
        grad_p=grad_p,
        hess_p=hess_p,
        Omega=Omega,
        kernel=kernel,
        constant_dim=len(target),
        kernel_dim=len(target),
        alpha=alpha,
        target=target,
        g=g,
        f=f,
        grad_f=grad_f,
        hess_f=hess_f,
        ssn_steps=1000,
    )

In [ ]:
def adapt_time(times, residuals, frame=100, resolution=1):
    to_return = []
    last_pos = 0
    last_res = residuals[0]
    for t in range(frame):
        for i, (res, tim) in enumerate(zip(residuals[last_pos:], times[last_pos:])):
            if tim > t+resolution:
                to_return.append(last_res)
                if tim  - t - resolution < resolution:
                    last_pos += i
                break
            else:
                last_res = res
    to_return.append(last_res)
    return to_return

def bring_to_same_length(arrays, mode):
    max_length = max(len(arr) for arr in arrays)
    new_arrays = []
    for arr in arrays:
        if len(arr) < max_length:
            if mode=="support":
                last_val = arr[-1]
            elif mode=="residual":
                last_val = 0
            arr = arr + [last_val] * (max_length - len(arr))
        new_arrays.append(arr)
    return new_arrays

In [ ]:
optimum = 2.19753851927879e-1

In [ ]:
logging.getLogger().setLevel(logging.CRITICAL) # Supress logging

In [ ]:
# deterministic NLGCG
u, c, times_det_nlgcg, supports_det_nlgcg, inner_loop, lgcg_lazy, lgcg_total, objective_values_det_nlgcg, dropped_tot, epsilons = exp_nlgcg.solve(tol=5e-14, max_radius=max_radius, temperature=0.1, mode="deterministic")
residuals_det_nlgcg = adapt_time(times_det_nlgcg, [obj - optimum for obj in objective_values_det_nlgcg], frame=1000, resolution=1)

In [ ]:
# Adaptive refinement
cells_dict, vertices_dict, vertices, u, objective_values_adaptive, times_adaptive, actives, supports_adaptive = exp_adaptive.solve(max_iters=200)
residuals_adaptive = adapt_time(times_adaptive, [obj - optimum for obj in objective_values_adaptive], frame=1000, resolution=1)

In [ ]:
# NLGCG stochastic
nlgcg_residuals = []
nlgcg_supports = []
for i in range(10):
    print(i)
    u, c, times_nlgcg, supports_nlgcg, inner_loop, lgcg_lazy, lgcg_total, objective_values_nlgcg, dropped_tot, epsilons = exp_nlgcg.solve(tol=5e-14, max_radius=max_radius, temperature=0.1)
    local_residuals = adapt_time(times_nlgcg, [obj - optimum for obj in objective_values_nlgcg], frame=1000, resolution=1)
    nlgcg_residuals.append(local_residuals)
    nlgcg_supports.append(supports_nlgcg)
nlgcg_residuals_mean = np.mean(bring_to_same_length(nlgcg_residuals, mode="residual"), axis=0)
nlgcg_residuals_std = np.std(bring_to_same_length(nlgcg_residuals, mode="residual"), axis=0)
nlgcg_supports_mean = np.mean(bring_to_same_length(nlgcg_supports, mode="support"), axis=0)
nlgcg_supports_std = np.std(bring_to_same_length(nlgcg_supports, mode="support"), axis=0)

In [ ]:
# Particle Descent Stochastic
particle_residuals = []
particle_supports = []
for i in range(10):
    print(i)
    u, c, objective_values_particle, supports_particle, times_particle, success = exp_particle.solve(max_iters=int(1e6), mode="uniform")
    local_residuals = adapt_time(times_particle, [obj - optimum for obj in objective_values_particle], frame=1000, resolution=1)
    particle_residuals.append(local_residuals)
    particle_supports.append(supports_particle)
particle_residuals_filtered = []
converged_frac = 0
for i in range(len(particle_residuals)):
    if particle_residuals[i][-1] < 1e-8:
        particle_residuals_filtered.append(particle_residuals[i])
        converged_frac += 0.1
particle_residuals_mean = np.mean(bring_to_same_length(particle_residuals_filtered, mode="residual"), axis=0)
particle_residuals_std = np.std(bring_to_same_length(particle_residuals_filtered, mode="residual"), axis=0)
particle_supports_mean = np.mean(bring_to_same_length(particle_supports, mode="support"), axis=0)
particle_supports_std = np.std(bring_to_same_length(particle_supports, mode="support"), axis=0)
print(converged_frac)

In [ ]:
# Plot residuals
fig, ax = plt.subplots(figsize=(5,5))
names = ["NLGCG", "Adaptive Refinement", "SNLGCG", "Particle Descent"]
styles = ["-", "-.", "--", ":"]
for array, name, style in zip([residuals_det_nlgcg, residuals_adaptive, nlgcg_residuals_mean, particle_residuals_mean], names, styles):
    ax.semilogy(np.arange(len(array)), array, style, label=name);
ax.fill(np.hstack((np.arange(len(particle_residuals_mean)), np.arange(len(particle_residuals_mean))[::-1])), np.hstack((np.array(particle_residuals_mean)-np.array(particle_residuals_std), np.array(particle_residuals_mean)[::-1]+np.array(particle_residuals_std)[::-1])), 'red', alpha=0.3);
ax.fill(np.hstack((np.arange(len(nlgcg_residuals_mean)), np.arange(len(nlgcg_residuals_mean))[::-1])), np.hstack((np.array(nlgcg_residuals_mean)-np.array(nlgcg_residuals_std), np.array(nlgcg_residuals_mean)[::-1]+np.array(nlgcg_residuals_std)[::-1])), 'green', alpha=0.3);
plt.ylabel("Objective residual");
plt.xlabel("Time (s)");
plt.ylim(1e-12, 100);
# plt.xlim(0, 100);
ax.legend();

In [ ]:
# Plot supports
fig, ax = plt.subplots(figsize=(5,5))
names = ["NLGCG", "Adaptive Refinement", "SNLGCG", "Particle Descent"]
styles = ["-", "-.", "--", ":"]
for array, name, style in zip([supports_det_nlgcg, supports_adaptive, nlgcg_supports_mean, particle_supports_mean], names, styles):
    ax.semilogx(np.arange(len(array)), array, style, label=name);
ax.fill(np.hstack((np.arange(len(particle_supports_mean)), np.arange(len(particle_supports_mean))[::-1])), np.hstack((np.array(particle_supports_mean)-np.array(particle_supports_std), np.array(particle_supports_mean)[::-1]+np.array(particle_supports_std)[::-1])), 'red', alpha=0.3);
ax.fill(np.hstack((np.arange(len(nlgcg_supports_mean)), np.arange(len(nlgcg_supports_mean))[::-1])), np.hstack((np.array(nlgcg_supports_mean)-np.array(nlgcg_supports_std), np.array(nlgcg_supports_mean)[::-1]+np.array(nlgcg_supports_std)[::-1])), 'green', alpha=0.3);
plt.ylabel("Support points");
plt.xlabel("Iterations");
# plt.ylim(1e-12, 100);
# plt.xlim(0, 100);
ax.legend();

In [ ]:
# Plot number of coefficients to potimize
fig, ax = plt.subplots(figsize=(5,5))
names = ["NLGCG", "Adaptive Refinement", "SNLGCG"]
styles = ["-", "-.", "--"]
for array, name, style in zip([supports_det_nlgcg, actives, nlgcg_supports_mean], names, styles):
    ax.semilogx(np.arange(len(array)), array, style, label=name);
ax.fill(np.hstack((np.arange(len(nlgcg_supports_mean)), np.arange(len(nlgcg_supports_mean))[::-1])), np.hstack((np.array(nlgcg_supports_mean)-np.array(nlgcg_supports_std), np.array(nlgcg_supports_mean)[::-1]+np.array(nlgcg_supports_std)[::-1])), 'green', alpha=0.3);
plt.ylabel("Number of coefficients to optimize");
plt.xlabel("Iterations");
# plt.ylim(1e-12, 100);
# plt.xlim(0, 100);
ax.legend();

## Plots

In [ ]:
a = np.arange(Omega[0][0],Omega[0][1],0.001)
p_u = p(u)
vals = p_u(a)
plt.plot(a,vals);
plt.axvline(x=-1, linestyle="-", c="r", label="Support");
plt.axvline(x=-1, linestyle="-", c="g", label="Truth");
for i, pos in enumerate(u.support):
    ymax = 0.5+(u.coefficients[i]*alpha*0.25+0.05*np.sign(u.coefficients[i]))
    plt.axvline(x=pos, ymin=0.5,ymax=ymax, linestyle="-", c="r");
for point, weight in  zip(true_sources, true_weights):
    ymax = 0.5+(weight*alpha*0.25+0.05*np.sign(weight))
    plt.axvline(x=point, ymin=0.5,ymax=ymax,alpha=0.5, linestyle="-", c="g");
plt.xlabel("Frequency");
plt.ylim(-alpha*1.1,alpha*1.1);
plt.xlim(0, 50);
plt.legend();

In [ ]:
# Plot the measured signal
def signal(x):
    # Input is 2D array of shape (number of points, Omega dimension)
    weighted_signal = 0
    for point, weight in zip(true_sources, true_weights):
        weighted_signal += weight*np.sin(2*np.pi*point*x).flatten()
    return weighted_signal

a = np.arange(0,1,0.001)
true_signal = signal(a)
plt.plot(a, true_signal);
plt.xlabel("Time");

In [ ]:
# Plot the measured signal
def signal(x):
    # Input is 2D array of shape (number of points, Omega dimension)
    weighted_signal = 0
    for point, weight in zip(u.support, u.coefficients):
        weighted_signal += weight*np.sin(2*np.pi*point*x).flatten()
    return weighted_signal

a = np.arange(0,1,0.001)
predicted_signal = signal(a)
error = true_signal - predicted_signal
print(f"L2 error: {np.linalg.norm(error)/np.sqrt(len(a))}, Linf error: {np.max(np.abs(error))}")
plt.plot(a, np.abs(true_signal-predicted_signal));
plt.xlabel("Time");

In [ ]:
# Plot residuals
residuals = np.array(objective_values) - 2.19753851927879e-1
plt.figure(figsize=(11.25,5))
plt.semilogy(np.array(range(len(residuals)-1)), residuals[:-1], linestyle="-.", color="green");
# for interval in intervals:
#     plt.fill_between(interval, 0, 60, hatch="/", color="gray", alpha=0.2);
plt.ylim(1e-17, 60);
# plt.xlim(2500, 3000);
plt.ylabel("Objective residual");
plt.xlabel("Total iterations");
plt.show()

# Function Approximation (Gaussian Shallow NN)

## Generate Data and Define Functions

In [ ]:
logging.getLogger().setLevel(logging.INFO) # Supress logging

In [ ]:
# sigma_space = np.array([[0,1]])
# omega_space = np.array([[0,1]])
# Omega = np.vstack((sigma_space, omega_space))
# d = omega_space.shape[0]
# variance_exponent = 0.1
# alpha = 1e-3
# observation_resolution = 20
# endpoint = True
# true_sigma = np.array([0.06])
# true_omega = np.array([[0.28]])
# true_sources = np.hstack((true_sigma.reshape(-1,1), true_omega))
# true_weights = np.array([1])
# true_measure = Measure(support=true_sources, coefficients=true_weights)
# method = "measure"

In [ ]:
# sigma_space = np.array([[0,1]])
# omega_space = np.array([[0,1]])
# Omega = np.vstack((sigma_space, omega_space))
# d = omega_space.shape[0]
# variance_exponent = 0.1
# alpha = 1e-3
# observation_resolution = 100
# endpoint = True
# true_sigma = np.array([0.06, 0.05, 0.04])
# true_omega = np.array([[0.28], [0.51], [0.71]])
# true_sources = np.hstack((true_sigma.reshape(-1,1), true_omega))
# true_weights = np.array([1, -0.7, 0.8])
# true_measure = Measure(support=true_sources, coefficients=true_weights)
# method = "measure"

In [ ]:
# sigma_space = np.array([[0,1]])
# omega_space = np.array([[0,1]])
# Omega = np.vstack((sigma_space, omega_space))
# d = omega_space.shape[0]
# variance_exponent = 0.1
# alpha = 1e-3
# observation_resolution = 100
# endpoint = True
# np.random.seed(49)
# true_sigma = np.random.rand(10)*0.1
# true_omega = (np.random.rand(10, 1)-np.array([0.5]))*0.9+np.array([0.5])
# true_sources = np.hstack((true_sigma.reshape(-1,1), true_omega))
# true_weights = np.random.rand(10) * 2 - 1
# true_measure = Measure(support=true_sources, coefficients=true_weights)
# method = "measure"

In [ ]:
# sigma_space = np.array([[0,1]])
# omega_space = np.array([[-1,1]])
# Omega = np.vstack((sigma_space, omega_space))
# d = omega_space.shape[0]
# variance_exponent = 0.1
# alpha = 1e-3
# observation_resolution = 100
# endpoint = True

# true_function = lambda x: sp.special.expit(100*x).flatten()
# method = "function"

In [ ]:
# sigma_space = np.array([0,1])
# omega_space = np.array([[0,1], [0,1]])
# Omega = np.vstack((sigma_space, omega_space))
# d = omega_space.shape[0]
# variance_exponent = 0.1
# alpha = 1e-3
# observation_resolution = 25
# endpoint = True
# true_sigma = np.array([0.06, 0.05, 0.04])
# true_omega = np.array([[0.28, 0.71], [0.51,0.27], [0.71,0.53]])
# true_sources = np.hstack((true_sigma.reshape(-1,1), true_omega))
# true_weights = np.array([1, -0.7, 0.8])
# true_measure = Measure(support=true_sources, coefficients=true_weights)
# method = "measure"

In [ ]:
# sigma_space = np.array([0,1])
# omega_space = np.array([[0,1], [0,1]])
# Omega = np.vstack((sigma_space, omega_space))
# d = omega_space.shape[0]
# variance_exponent = 0.1
# alpha = 5e-3
# observation_resolution = 25
# endpoint = True
# np.random.seed(49)
# true_sigma = np.random.rand(20)*0.2
# true_omega = (np.random.rand(20, 2)-np.array([0.5,0.5]))*0.9+np.array([0.5,0.5])
# true_sources = np.hstack((true_sigma.reshape(-1,1), true_omega))
# true_weights = np.random.rand(20) * 2 - 1
# true_measure = Measure(support=true_sources, coefficients=true_weights)
# method = "measure"

In [ ]:
# sigma_space = np.array([0,1])
# omega_space = np.array([[-1,1], [-1,1]])
# Omega = np.vstack((sigma_space, omega_space))
# d = omega_space.shape[0]
# variance_exponent = 0.1
# alpha = 4e-3
# observation_resolution = 25
# endpoint = True

# def my_funct(x: np.ndarray, c: np.ndarray, k: float, r: float):
#     return np.tanh(k*(r-np.linalg.norm(x-c,axis=1)))+1

# c_1 = np.array([0.3,0.3])
# k_1 = 4
# r_1 = 0.3
# c_2 = -c_1
# k_2 = 12
# r_2 = 0.15
# true_function = lambda x: my_funct(x, c_1, k_1, r_1) + my_funct(x, c_2, k_2, r_2)
# method = "function"

In [ ]:
sigma_space = np.array([0,2])
omega_space = np.array([[-1,1], [-1,1]])
Omega = np.vstack((sigma_space, omega_space))
d = omega_space.shape[0]
variance_exponent = 0.1
alpha = 1e-3 # 1e-4
observation_resolution = 20
endpoint = False
max_radius = 1e-2 # 1e-3
max_inner_loop = 25
cg_lambda = 1e-3
cg_iterations = 100
# 7 min
# 0.0036720086901201677, 129


true_function = lambda x: np.minimum(1-np.abs(x[:,0]), 1-np.abs(x[:,1]))
method = "function"

In [ ]:
# sigma_space = np.array([0,1])
# omega_space = np.array([[-1,1], [-1,1], [-1,1], [-1,1]])
# Omega = np.vstack((sigma_space, omega_space))
# d = omega_space.shape[0]
# variance_exponent = 0.1
# alpha = 1e-3  #49 support
# observation_resolution = 5
# endpoint = False
# max_radius = 1e-2
# max_inner_loop = 25
# cg_lambda = 1e-3
# cg_iterations = 100

# def my_funct(x: np.ndarray):
#     to_return = np.ones(x.shape[0])
#     for i in range(x.shape[1]):
#         to_return *= np.sin(np.pi*x[:,i])
#     return to_return

# true_function = lambda x: my_funct(x)
# method = "function"

In [ ]:
max_radius = (Omega[0][1]-Omega[0][0])/100

In [ ]:
observations = (np.array(np.meshgrid(
                    *(
                        np.linspace(bound[0]+1e-5, bound[1]-np.sqrt(3)*1e-5, observation_resolution, endpoint=endpoint)
                        for bound in omega_space
                    ))
            ).reshape(len(omega_space), -1).T)
# observations = np.array([obs for obs in observations if all(obs!=0) and all(obs!=1)])

In [ ]:
@jax.jit
def singleton_kernel(omega: np.ndarray):
    sigma = omega[0]
    x = omega[1:]
    inner = -jnp.sum((x - observations)**2,axis=1)/(2*sigma**2)
    outer = (jnp.exp(inner)*sigma**(variance_exponent))/(jnp.sqrt(2*jnp.pi)**d)
    return outer

grad_kernel = jax.jit(jax.jacobian(singleton_kernel))
hess_kernel = jax.jit(jax.hessian(singleton_kernel))
_ = grad_kernel(np.ones(len(Omega)))
_ = hess_kernel(np.ones(len(Omega)))

kernel = jax.vmap(singleton_kernel)
grad_kernel = jax.vmap(grad_kernel)
hess_kernel = jax.vmap(hess_kernel)

In [ ]:
if method == "measure":
    target = true_measure.duality_pairing(kernel)
elif method == "function":
    target = true_function(observations)

In [ ]:
@jax.jit
def g(w: np.ndarray) -> float:
    return alpha * jnp.linalg.norm(w, ord=1)
grad_g = jax.jit(jax.grad(g))
_ = grad_g(jnp.ones(1))

@jax.jit
def f(y: np.ndarray) -> float:
    return 0.5 * jnp.sum((y - target)**2)
grad_f = jax.jit(jax.grad(f))
hess_f = jax.jit(jax.hessian(f))
_ = grad_f(jnp.zeros(target.shape[0]))
_ = hess_f(jnp.zeros(target.shape[0]))

j = lambda u, c: f(u.duality_pairing(kernel)+c*np.ones(target.shape)) + g(u.coefficients)

In [ ]:
# def p(u, c):
#     inner = -grad_f(u.duality_pairing(kernel, len(target))+c*np.ones(target.shape))
#     return lambda omega: kernel(omega) @ inner

# def grad_p(u, c):
#     inner = -grad_f(u.duality_pairing(kernel, len(target))+c*np.ones(target.shape))
#     return lambda omega: np.tensordot(grad_kernel(omega), inner, axes=([1,0]))

# def hess_p(u, c):
#     inner = -grad_f(u.duality_pairing(kernel, len(target))+c*np.ones(target.shape))
#     return lambda omega: np.tensordot(hess_kernel(omega), inner, axes=([1,0]))

In [ ]:
@jax.jit
def p_raw(parameters: np.ndarray, c: float, omega: np.ndarray):
    coefficients = parameters[:,0]
    support = parameters[:,1:]
    Ku = jnp.tensordot(kernel(support), coefficients, axes=([0], [0]))
    constant_term = c*jnp.ones(len(target))
    return singleton_kernel(omega) @ -grad_f(Ku+constant_term)

grad_p_raw = jax.jit(jax.grad(p_raw, argnums=2))
hess_p_raw = jax.jit(jax.hessian(p_raw, argnums=2))

p_raw = jax.jit(jax.vmap(p_raw, in_axes=(None, None, 0), out_axes=0))
grad_p_raw = jax.jit(jax.vmap(grad_p_raw, in_axes=(None, None, 0), out_axes=0))
hess_p_raw = jax.jit(jax.vmap(hess_p_raw, in_axes=(None, None, 0), out_axes=0))

p = lambda u, c: lambda omega: p_raw(u.to_matrix(len(Omega)), c, omega)
grad_p = lambda u, c: lambda omega: grad_p_raw(u.to_matrix(len(Omega)), c, omega)
hess_p = lambda u, c: lambda omega: hess_p_raw(u.to_matrix(len(Omega)), c, omega)

In [ ]:
# Parameterized versions of f and j
@jax.jit
def f_N(raw_input: np.ndarray) -> float:
    constant = raw_input[-1]
    input = raw_input[:-1].reshape(-1, d+2)
    weights = input[:,0]
    omega = input[:,1:]
    return f(kernel(omega).T@weights+constant*jnp.ones(target.shape))

@jax.jit
def j_N(raw_input: np.ndarray) -> float:
    input = raw_input[:-1].reshape(-1, d+2)
    weights = input[:,0]
    return f_N(raw_input) + g(weights)

grad_f_N = jax.jit(jax.grad(f_N))
hess_f_N = jax.jit(jax.hessian(f_N))
grad_j_N = jax.jit(jax.grad(j_N))
hess_j_N = jax.jit(jax.hessian(j_N))

In [ ]:
# Define functions where the derivatives are taken wrt regularized (weights) and non-regularized(support + constant) parameters

@jax.jit
def f_N_(weights: np.ndarray, support_constant: np.ndarray) -> float:
    constant = support_constant[-1]
    omega = support_constant[:-1].reshape(-1, d+1)
    return f(kernel(omega).T@weights+constant*jnp.ones(target.shape))

@jax.jit
def j_N_(weights: np.ndarray, support_constant: np.ndarray) -> float:
    return f_N_(weights, support_constant) + g(weights)

grad_f_N_reg = jax.jit(jax.grad(f_N_, argnums=0))
hess_f_N_reg = jax.jit(jax.hessian(f_N_, argnums=0))
grad_j_N_reg = jax.jit(jax.grad(j_N_, argnums=0))
hess_j_N_reg = jax.jit(jax.hessian(j_N_, argnums=0))

grad_f_N_non_reg = jax.jit(jax.grad(f_N_, argnums=1))
hess_f_N_non_reg = jax.jit(jax.hessian(f_N_, argnums=1))
grad_j_N_non_reg = jax.jit(jax.grad(j_N_, argnums=1))
hess_j_N_non_reg = jax.jit(jax.hessian(j_N_, argnums=1))

## Experiments

### NLGCG

In [ ]:
exp = NLGCG(target=target, 
           kernel=kernel, 
           g=g,
           f=f,
           f_N=f_N,
           grad_f=grad_f,
           hess_f=hess_f,
           grad_f_N=grad_f_N,
           hess_f_N=hess_f_N,
           grad_f_N_non_reg=grad_f_N_non_reg,
           j=j,
           j_N=j_N,
           j_N_=j_N_,
           p=p,
           grad_p=grad_p,
           hess_p=hess_p,
           grad_j_N=grad_j_N,
           hess_j_N=hess_j_N,
           alpha=alpha,
           Omega=Omega,
           global_search_resolution=5,
           dual_variable_goodness=0.3,
           constant_dim=len(target),
           kernel_dim=len(target),
           newton_tolerance=2e-2
           )

In [ ]:
u, c, times, supports, inner_loop, lgcg_lazy, lgcg_total, objective_values, dropped_tot, epsilons = exp.solve(tol=5e-14, max_radius=max_radius, temperature=0.1)

In [ ]:
params, u_ks, radii = exp.local_merging_update_radii(u_, c_)
full_parameters = np.hstack((params.flatten(), c_))

In [ ]:
full_parameters = np.hstack((u.to_matrix().flatten(), c))

In [ ]:
grd = exp.grad_j_N(full_parameters)
np.linalg.norm(grd)

In [ ]:
hss = exp.hess_j_N(full_parameters)
np.linalg.eigvals(hss)

In [ ]:
print(u)

In [ ]:
c

### Particle Gradient Descent

In [ ]:
exp = ParticleDescent(
        m=2000,
        j=j,
        p=p,
        grad_p=grad_p,
        Omega=Omega,
        a_parameter=0.01,
        b_parameter=0.01/2,
        kernel=kernel,
        constant_dim=len(target),
        kernel_dim=len(target),
        alpha=alpha,
        target=target,
        g=g,
        f=f,
        grad_f=grad_f,
        hess_f=hess_f,
        ssn_steps=100,
    )

In [ ]:
u, c, objective_values_particle, supports_particle, times_particle, success = exp.solve(max_iters=int(1e6), max_time=10000)

In [ ]:
[float(v) for v in objective_values[-100:]]

In [ ]:
full_parameters = np.hstack((u.to_matrix().flatten(), c))
grd = exp.grad_j_N(full_parameters)
np.linalg.norm(grd)

In [ ]:
print(u)

### Adaptive Refinement

In [ ]:
exp = AdaptiveRefinement(
        observations=observations,
        variance_exponent=variance_exponent,
        j=j,
        p=p,
        grad_p=grad_p,
        hess_p=hess_p,
        Omega=Omega,
        kernel=kernel,
        constant_dim=len(target),
        kernel_dim=len(target),
        alpha=alpha,
        target=target,
        g=g,
        f=f,
        grad_f=grad_f,
        hess_f=hess_f,
        ssn_steps=1000,
    )

In [ ]:
cells_dict, vertices_dict, vertices, u, objective_values_adaptive, times_adaptive, actives, supports_adaptive = exp.solve(max_iters=100, max_time=1000)

In [ ]:
print(u)

### Full

In [ ]:
exp_nlgcg = NLGCG(target=target, 
           kernel=kernel, 
           g=g,
           f=f,
           f_N=f_N,
           grad_f=grad_f,
           hess_f=hess_f,
           grad_f_N=grad_f_N,
           hess_f_N=hess_f_N,
           grad_f_N_non_reg=grad_f_N_non_reg,
           j=j,
           j_N=j_N,
           j_N_=j_N_,
           p=p,
           grad_p=grad_p,
           hess_p=hess_p,
           grad_j_N=grad_j_N,
           hess_j_N=hess_j_N,
           alpha=alpha,
           Omega=Omega,
           global_search_resolution=10,
           dual_variable_goodness=0.3,
           constant_dim=len(target),
           kernel_dim=len(target),
           newton_tolerance=2e-2
           )

In [ ]:
exp_particle = ParticleDescent(
        m=1000,
        j=j,
        p=p,
        grad_p=grad_p,
        Omega=Omega,
        a_parameter=0.01,
        b_parameter=0.01/2,
        kernel=kernel,
        constant_dim=len(target),
        kernel_dim=len(target),
        alpha=alpha,
        target=target,
        g=g,
        f=f,
        grad_f=grad_f,
        hess_f=hess_f,
        ssn_steps=100,
    )

In [ ]:
exp_adaptive = AdaptiveRefinement(
        observations=observations,
        variance_exponent=variance_exponent,
        j=j,
        p=p,
        grad_p=grad_p,
        hess_p=hess_p,
        Omega=Omega,
        kernel=kernel,
        constant_dim=len(target),
        kernel_dim=len(target),
        alpha=alpha,
        target=target,
        g=g,
        f=f,
        grad_f=grad_f,
        hess_f=hess_f,
        ssn_steps=1000,
    )

In [ ]:
def adapt_time(times, residuals, frame=100, resolution=1):
    to_return = []
    last_pos = 0
    last_res = residuals[0]
    for t in range(frame):
        for i, (res, tim) in enumerate(zip(residuals[last_pos:], times[last_pos:])):
            if tim > t+resolution:
                to_return.append(last_res)
                if tim  - t - resolution < resolution:
                    last_pos += i
                break
            else:
                last_res = res
    to_return.append(last_res)
    return to_return

def bring_to_same_length(arrays, mode):
    max_length = max(len(arr) for arr in arrays)
    new_arrays = []
    for arr in arrays:
        if len(arr) < max_length:
            if mode=="support":
                last_val = arr[-1]
            elif mode=="residual":
                last_val = 0
            arr = arr + [last_val] * (max_length - len(arr))
        new_arrays.append(arr)
    return new_arrays

In [ ]:
optimum = 3.39493125929729e-02
logging.getLogger().setLevel(logging.CRITICAL) # Supress logging

In [ ]:
# deterministic NLGCG
u, c, times_det_nlgcg, supports_det_nlgcg, inner_loop, lgcg_lazy, lgcg_total, objective_values_det_nlgcg, dropped_tot, epsilons = exp_nlgcg.solve(tol=5e-14, max_radius=max_radius, temperature=0.1, mode="deterministic")
residuals_det_nlgcg = adapt_time(times_det_nlgcg, [obj - optimum for obj in objective_values_det_nlgcg], frame=1500, resolution=1)

In [ ]:
residuals_det_nlgcg

In [ ]:
# Adaptive refinement
cells_dict, vertices_dict, vertices, u, objective_values_adaptive, times_adaptive, actives, supports_adaptive = exp_adaptive.solve(max_iters=1000, max_time=1500)
residuals_adaptive = adapt_time(times_adaptive, [obj - optimum for obj in objective_values_adaptive], frame=1500, resolution=1)

In [ ]:
# NLGCG stochastic
nlgcg_residuals = []
nlgcg_supports = []
for i in range(10):
    print(i)
    u, c, times_nlgcg, supports_nlgcg, inner_loop, lgcg_lazy, lgcg_total, objective_values_nlgcg, dropped_tot, epsilons = exp_nlgcg.solve(tol=5e-14, max_radius=max_radius, temperature=0.1)
    local_residuals = adapt_time(times_nlgcg, [obj - optimum for obj in objective_values_nlgcg], frame=1500, resolution=1)
    nlgcg_residuals.append(local_residuals)
    nlgcg_supports.append(supports_nlgcg)
nlgcg_residuals_mean = np.mean(bring_to_same_length(nlgcg_residuals, mode="residual"), axis=0)
nlgcg_residuals_std = np.std(bring_to_same_length(nlgcg_residuals, mode="residual"), axis=0)
nlgcg_supports_mean = np.mean(bring_to_same_length(nlgcg_supports, mode="support"), axis=0)
nlgcg_supports_std = np.std(bring_to_same_length(nlgcg_supports, mode="support"), axis=0)

In [ ]:
# Particle Descent Stochastic
particle_residuals = []
particle_supports = []
for i in range(10):
    print(i)
    u, c, objective_values_particle, supports_particle, times_particle, success = exp_particle.solve(max_iters=int(1e6), max_time=1500)
    local_residuals = adapt_time(times_particle, [obj - optimum for obj in objective_values_particle], frame=1000, resolution=1)
    particle_residuals.append(local_residuals)
    particle_supports.append(supports_particle)
particle_residuals_filtered = []
converged_frac = 0
for i in range(len(particle_residuals)):
    if particle_residuals[i][-1] < 1e-6:
        particle_residuals_filtered.append(particle_residuals[i])
        converged_frac += 0.1
particle_residuals_mean = np.mean(bring_to_same_length(particle_residuals_filtered, mode="residual"), axis=0)
particle_residuals_std = np.std(bring_to_same_length(particle_residuals_filtered, mode="residual"), axis=0)
particle_supports_mean = np.mean(bring_to_same_length(particle_supports, mode="support"), axis=0)
particle_supports_std = np.std(bring_to_same_length(particle_supports, mode="support"), axis=0)
print(converged_frac)

In [ ]:
# Plot residuals
fig, ax = plt.subplots(figsize=(5,5))
names = ["NLGCG", "Adaptive Refinement", "SNLGCG", "Particle Descent"]
styles = ["-", "-.", "--", ":"]
for array, name, style in zip([residuals_det_nlgcg, residuals_adaptive, nlgcg_residuals_mean, particle_residuals_mean], names, styles):
    ax.semilogy(np.arange(len(array)), array, style, label=name);
ax.fill(np.hstack((np.arange(len(particle_residuals_mean)), np.arange(len(particle_residuals_mean))[::-1])), np.hstack((np.array(particle_residuals_mean)-np.array(particle_residuals_std), np.array(particle_residuals_mean)[::-1]+np.array(particle_residuals_std)[::-1])), 'red', alpha=0.3);
ax.fill(np.hstack((np.arange(len(nlgcg_residuals_mean)), np.arange(len(nlgcg_residuals_mean))[::-1])), np.hstack((np.array(nlgcg_residuals_mean)-np.array(nlgcg_residuals_std), np.array(nlgcg_residuals_mean)[::-1]+np.array(nlgcg_residuals_std)[::-1])), 'green', alpha=0.3);
plt.ylabel("Objective residual");
plt.xlabel("Time (s)");
plt.ylim(1e-12, 100);
# plt.xlim(0, 100);
ax.legend();

In [ ]:
# Plot supports
fig, ax = plt.subplots(figsize=(5,5))
names = ["NLGCG", "Adaptive Refinement", "SNLGCG", "Particle Descent"]
styles = ["-", "-.", "--", ":"]
for array, name, style in zip([supports_det_nlgcg, supports_adaptive, nlgcg_supports_mean, particle_supports_mean], names, styles):
    ax.semilogx(np.arange(len(array)), array, style, label=name);
ax.fill(np.hstack((np.arange(len(particle_supports_mean)), np.arange(len(particle_supports_mean))[::-1])), np.hstack((np.array(particle_supports_mean)-np.array(particle_supports_std), np.array(particle_supports_mean)[::-1]+np.array(particle_supports_std)[::-1])), 'red', alpha=0.3);
ax.fill(np.hstack((np.arange(len(nlgcg_supports_mean)), np.arange(len(nlgcg_supports_mean))[::-1])), np.hstack((np.array(nlgcg_supports_mean)-np.array(nlgcg_supports_std), np.array(nlgcg_supports_mean)[::-1]+np.array(nlgcg_supports_std)[::-1])), 'green', alpha=0.3);
plt.ylabel("Support points");
plt.xlabel("Iterations");
# plt.ylim(1e-12, 100);
# plt.xlim(0, 100);
ax.legend();

In [ ]:
# Plot number of coefficients to potimize
fig, ax = plt.subplots(figsize=(5,5))
names = ["NLGCG", "Adaptive Refinement", "SNLGCG"]
styles = ["-", "-.", "--"]
for array, name, style in zip([supports_det_nlgcg, actives, nlgcg_supports_mean], names, styles):
    ax.semilogx(np.arange(len(array)), array, style, label=name);
ax.fill(np.hstack((np.arange(len(nlgcg_supports_mean)), np.arange(len(nlgcg_supports_mean))[::-1])), np.hstack((np.array(nlgcg_supports_mean)-np.array(nlgcg_supports_std), np.array(nlgcg_supports_mean)[::-1]+np.array(nlgcg_supports_std)[::-1])), 'green', alpha=0.3);
plt.ylabel("Number of coefficients to optimize");
plt.xlabel("Iterations");
# plt.ylim(1e-12, 100);
# plt.xlim(0, 100);
ax.legend();

## Full Tests

In [ ]:
def define_particle_descent(m, a, b):
    return ParticleDescent(
        m=m,
        j=j,
        p=p,
        grad_p=grad_p,
        Omega=Omega,
        a_parameter=a,
        b_parameter=b,
        kernel=kernel,
        constant_dim=len(target),
        kernel_dim=len(target),
        alpha=alpha,
        target=target,
        g=g,
        f=f,
        grad_f=grad_f,
        hess_f=hess_f,
        ssn_steps=100,
    )

def perform_experiment(m, quotient):
    a = 1
    b = a/quotient
    condition = True
    while condition:
        exp = define_particle_descent(m, a, b)
        u, c, objective_values, times, success = exp.solve(max_iters=int(1e6), max_time=1000)
        obj = objective_values[-1]
        if not success:
            a *= 0.5
            b = a/quotient
        else:
            condition = False
        del exp
    return obj, a

def grid_search_particle_descent(m_values, quotient_values):
    results = {}
    for m in m_values:
        for quotient in quotient_values:
            found_values = []
            for _ in range(10):
                obj, a = perform_experiment(m, quotient)
                logging.info(f"m={m}, quotient={quotient}, obj={obj}, a={a}")
                found_values.append((obj, a))
            results[(m, quotient)] = found_values
    return results

In [ ]:
# logging.getLogger().setLevel(logging.CRITICAL) # Supress logging
results_dict = grid_search_particle_descent(m_values=[25, 40], quotient_values=[2])
# logging.getLogger().setLevel(logging.DEBUG) # Activate logging

In [ ]:
results_dict

In [ ]:
opt = 0.00249611254129951
clean_results = {}
for key, val in results_dict.items():
    success_ratio = 0
    average_param = 0
    for v in val:
        obj, a = v
        average_param += 0.1*a
        if abs(obj-opt) < 1e-10:
            success_ratio += 0.1
    clean_results[key] = (success_ratio, average_param)

In [ ]:
clean_results

In [ ]:
clean_results

## Plots

In [ ]:
# Dual variable in 1D omega space
p_u = p(u, c)
resolution = 100
a_sigma = np.linspace(0,0.25,resolution, endpoint=False)
a_omega = np.linspace(omega_space[0][0],omega_space[0][1],resolution, endpoint=False)
x, y = np.meshgrid(a_sigma,a_omega)
points = np.array(list(zip(x.flatten(), y.flatten())))
vals = np.abs(p_u(points)).reshape((resolution,resolution))

plt.contourf(x, y, vals, levels=100);
for omega in u.support:
    plt.plot(omega[0], omega[1], "o", c="black", markersize=5);
plt.colorbar();
# plt.xlim(0,0.5)

In [ ]:
p_u = p(u, c)
p_u(u.support)

In [ ]:
# Projection of the dual variable on an axis
p_u = p(u, c)
resolution = 100
a_sigma = np.linspace(0,1,resolution, endpoint=False)
index = 11
points = np.array([u.support[index]]*resolution)
points[:,0]=a_sigma
vals = np.abs(p_u(points))

plt.plot(a_sigma, vals);
plt.plot([u.support[index][0]], np.abs(p_u(np.array([u.support[index]]))), "o", c="black", markersize=5);

In [ ]:
# Parameterized objective, projected into 2D
point = np.array([1,0.3,0.3,0.3,0.1]) # u.to_matrix().flatten()
resolution = 100
a_1 = np.linspace(0,1,resolution, endpoint=False)
a_2 = np.linspace(0,1,resolution, endpoint=False)
x, y = np.meshgrid(a_1,a_2)
raw_points = np.array(list(zip(x.flatten(), y.flatten())))
points = np.array([point]*len(raw_points))
points[:,2:4] = raw_points

# points = np.hstack((0.02*np.ones((points.shape[0],1)), points))
vals = np.log(jax.vmap(j_N)(points)).reshape((resolution,resolution))

plt.contourf(x, y, vals, levels=100);
plt.colorbar();

In [ ]:
# 1D problem with both true and predicted sources
a = np.arange(0,1,0.01).reshape(-1,1)

def plot_kernel(omega: np.ndarray):
    sigma = omega[0]
    x_omega = omega[1:]
    inner = -(jnp.linalg.norm(x_omega - a,axis=1)**2)/(2*sigma**2)
    outer = (jnp.exp(inner)*sigma**(variance_exponent))/(np.sqrt(2*np.pi)**d)
    return outer

true_vals = true_measure.duality_pairing(jax.vmap(plot_kernel)) # .reshape((100,100))
pred_vals = u.duality_pairing(jax.vmap(plot_kernel)) + c*np.ones(len(a))

plt.plot(a, true_vals, c="blue")
plt.plot(a, pred_vals, c="red");
plt.xlim(0,1);
plt.ylim(-1, 1);
signs = np.sign(true_weights)
for i, x in enumerate(true_sources):
    size = signs[i]*(0.01+np.sqrt(np.abs(true_weights[i]))*0.3)+0.5
    plt.axvline(x=x[1], ymin=min(0.5,size)+0.01,ymax=max(0.5, size)-0.01, linewidth=5, c="blue", alpha=sp.special.expit(np.log(np.sqrt(x[0]*5))));
signs = np.sign(u.coefficients)
for i, x in enumerate(u.support):
    size = signs[i]*(0.01+np.sqrt(np.abs(u.coefficients[i]))*0.3)+0.5
    plt.axvline(x=x[1], ymin=min(0.5,size)+0.01,ymax=max(0.5,size)-0.01, linewidth=5, c="red", alpha=sp.special.expit(np.log(np.sqrt(x[0]*5))));

In [ ]:
# 1D problem with a true function and predicted sources
resolution = 1000
a = np.linspace(omega_space[0][0],omega_space[0][1],resolution,endpoint=False).reshape(-1,1)

def plot_kernel(omega: np.ndarray):
    sigma = omega[0]
    x_omega = omega[1:]
    inner = -(jnp.linalg.norm(x_omega - a,axis=1)**2)/(2*sigma**2)
    outer = (jnp.exp(inner)*sigma**(variance_exponent))/(np.sqrt(2*np.pi)**d)
    return outer

true_vals = true_function(a)
pred_vals = u.duality_pairing(jax.vmap(plot_kernel)) + c*np.ones(len(a))

plt.plot(a, true_vals, c="blue")
plt.plot(a, pred_vals, c="red");
plt.xlim(omega_space[0][0],omega_space[0][1]);
signs = np.sign(u.coefficients)
for i, x in enumerate(u.support):
    if signs[i]>0:
        color = "blue"
    else:
        color = "red"
    size = (0.01+np.sqrt(np.abs(u.coefficients[i]))*0.5)
    plt.axvline(x=x[1], ymin=0.01,ymax=size-0.01, linewidth=5, c=color, alpha=sp.special.expit(np.log(np.sqrt(x[0]*5))));

In [ ]:
# 2D problem with both true and predicted sources
resolution = 100
a_1 = np.linspace(omega_space[0][0],omega_space[0][1],resolution, endpoint=False)
a_2 = np.linspace(omega_space[1][0],omega_space[1][1],resolution, endpoint=False)
x, y = np.meshgrid(a_1,a_2)
points = np.array(list(zip(x.flatten(), y.flatten())))

def plot_kernel(omega: np.ndarray):
    sigma = omega[0]
    x_omega = omega[1:]
    inner = -(jnp.linalg.norm(x_omega - points,axis=1)**2)/(2*sigma**2)
    outer = (jnp.exp(inner)*sigma**variance_exponent)/(np.sqrt(2*np.pi)**d)
    return outer

true_vals = true_measure.duality_pairing(jax.vmap(plot_kernel)).reshape((100,100))
pred_vals = u.duality_pairing(jax.vmap(plot_kernel)).reshape((100,100))

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15,4))
contour1 = ax1.contourf(x, y, true_vals, levels=100)
fig.colorbar(contour1, ax=ax1)
contour2 = ax2.contourf(x, y, pred_vals, levels=100)
fig.colorbar(contour2, ax=ax2)
contour3 = ax3.contourf(x, y, np.abs(pred_vals-true_vals), levels=100)
fig.colorbar(contour3, ax=ax3)
for i, x in enumerate(true_sources):
    if true_weights[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(true_weights[i])*10
    ax1.plot([x[1]], [x[2]], "P", c=color, markersize=size, alpha=0.5);
    ax1.add_patch(plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5))

for i, x in enumerate(u.support):
    if u.coefficients[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(u.coefficients[i])*10
    ax2.plot([x[1]], [x[2]], "o", c=color, markersize=size, alpha=0.5);
    ax2.add_patch(plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5))
    ax3.plot([x[1]], [x[2]], "o", c=color, markersize=size, alpha=0.5);
    ax3.add_patch(plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5))

ax1.set_xlim(omega_space[0][0],omega_space[0][1])
ax1.set_ylim(omega_space[1][0],omega_space[1][1])
ax2.set_xlim(omega_space[0][0],omega_space[0][1])
ax2.set_ylim(omega_space[1][0],omega_space[1][1])
ax3.set_xlim(omega_space[0][0],omega_space[0][1])
ax3.set_ylim(omega_space[1][0],omega_space[1][1])

In [ ]:
# 2D problem with a true function and predicted sources
resolution = 100
a_1 = np.linspace(omega_space[0][0],omega_space[0][1],resolution, endpoint=False)
a_2 = np.linspace(omega_space[1][0],omega_space[1][1],resolution, endpoint=False)
x, y = np.meshgrid(a_1,a_2)
points = np.array(list(zip(x.flatten(), y.flatten())))
vals = true_function(points).reshape((resolution,resolution))

def plot_kernel(omega: np.ndarray):
    sigma = omega[0]
    x_omega = omega[1:]
    inner = -(jnp.linalg.norm(x_omega - points,axis=1)**2)/(2*sigma**2)
    outer = (jnp.exp(inner)*sigma**variance_exponent)/(np.sqrt(2*np.pi)**d)
    return outer

pred_vals = u.duality_pairing(jax.vmap(plot_kernel)).reshape((resolution,resolution)) +c*np.ones((resolution, resolution))

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15,4))
contour1 = ax1.contourf(x, y, vals, levels=100)
fig.colorbar(contour1, ax=ax1)
contour2 = ax2.contourf(x, y, pred_vals, levels=100)
fig.colorbar(contour2, ax=ax2)
contour3 = ax3.contourf(x, y, np.abs(pred_vals-vals), levels=100)
fig.colorbar(contour3, ax=ax3)
for i, x in enumerate(u.support):
    if u.coefficients[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(u.coefficients[i])
    # ax1.add_patch(plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5))
    ax2.add_patch(plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5))
    ax3.add_patch(plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5))
ax1.set_xlim(omega_space[0][0],omega_space[0][1])
ax1.set_ylim(omega_space[1][0],omega_space[1][1])
ax2.set_xlim(omega_space[0][0],omega_space[0][1])
ax2.set_ylim(omega_space[1][0],omega_space[1][1])
ax3.set_xlim(omega_space[0][0],omega_space[0][1])
ax3.set_ylim(omega_space[1][0],omega_space[1][1])

logging.info(f"L2: {np.linalg.norm(pred_vals-vals)/len(vals)}, Linf: {np.max(np.abs(pred_vals-vals))}")

In [ ]:
# Higher-dimensional problem
resolution = 20
a_1 = np.linspace(omega_space[0][0],omega_space[0][1],resolution, endpoint=False)
a_2 = np.linspace(omega_space[1][0],omega_space[1][1],resolution, endpoint=False)
a_3 = np.linspace(omega_space[2][0],omega_space[2][1],resolution, endpoint=False)
a_4 = np.linspace(omega_space[3][0],omega_space[3][1],resolution, endpoint=False)
w, x, y, z = np.meshgrid(a_1,a_2,a_3,a_4)
points = np.array(list(zip(w.flatten(), x.flatten(), y.flatten(), z.flatten())))
vals = true_function(points)

def plot_kernel(omega: np.ndarray):
    sigma = omega[0]
    x_omega = omega[1:]
    inner = -(jnp.linalg.norm(x_omega - points,axis=1)**2)/(2*sigma**2)
    outer = (jnp.exp(inner)*sigma**variance_exponent)/(np.sqrt(2*np.pi)**d)
    return outer

pred_vals = u.duality_pairing(jax.vmap(plot_kernel)) +c*np.ones(points.shape[0])

err = np.abs(vals-pred_vals)
print(f"L2 error: {np.linalg.norm(err)}, Linf error: {np.max(err)}, average error: {np.mean(err)}")

In [ ]:
# Convergence in iterations with shaded inner loops
# intervals = []
# current_inner = False
# for ind, i in enumerate(inner_loop):
#     if i:
#         if not current_inner:
#             start = ind-0.5
#             current_inner = True
#     else:
#         if current_inner:
#             end = ind-0.5
#             intervals.append((start,end))
#             current_inner = False
# if inner_loop[-1]:
#     end = len(inner_loop)-0.5
#     intervals.append((start,end))

residuals = np.array(objective_values) - np.min(objective_values)
plt.figure(figsize=(11.25,5))
plt.semilogy(np.array(range(len(residuals)-1)), residuals[:-1], linestyle="-.", color="green");
# for interval in intervals:
#     plt.fill_between(interval, 0, 60, hatch="/", color="gray", alpha=0.2);
plt.ylim(1e-17, 60);
# plt.xlim(2500, 3000);
plt.ylabel("Objective residual");
plt.xlabel("Total iterations");
plt.show()

In [ ]:
# Dual variable in 1D omega space for adaptive refinement
p_u = p(u, 0)
grad_p_u = grad_p(u, 0)
hess_p_u = hess_p(u, 0)
resolution = 100
a_sigma = np.linspace(0,1,resolution, endpoint=False)
a_omega = np.linspace(omega_space[0][0],omega_space[0][1],resolution, endpoint=False)
x, y = np.meshgrid(a_sigma,a_omega)
points = np.array(list(zip(x.flatten(), y.flatten())))
vals = np.abs(p_u(points)).reshape((resolution,resolution))

plt.contourf(x, y, vals, levels=100);
for vertex in vertices:
    plt.plot(vertex[0], vertex[1], "o", c="red", markersize=0.2);
plt.colorbar();
plt.xlim(0.055,0.065)
plt.ylim(0.275,0.285)

In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(5,5))
for i, x in enumerate(poss[:,1:]):
    if vals[i]<1e-7:
        alpha = 0.2
    elif vals[i]<1e-6:
        alpha = 0.3
    elif vals[i]<1e-5:
        alpha = 0.4
    elif vals[i]<1e-4:
        alpha=1
    # ax1.plot(x[1], x[2], "o", c="r", markersize=5, alpha=alpha);
    ax1.add_patch(plt.Circle((x[1], x[2]), radius=x[0], color="r", fill=False, alpha=alpha))
ax1.set_xlim(omega_space[0][0],omega_space[0][1])
ax1.set_ylim(omega_space[1][0],omega_space[1][1])

In [ ]:
nlgcg_obj = objective_values
nlgcg_times = times

In [ ]:
best_val = min(np.min(nlgcg_obj), np.min(chizat_obj), np.min(flinth_obj))
names = ["NLGCG", "Particle Descent", "Adaptive Refinement"]
styles = ["-", "-.", "--"]
plt.figure(figsize=(9,5))
for array, ts, name, style in zip([nlgcg_obj, chizat_obj[:-1], flinth_obj], [nlgcg_times, chizat_times, flinth_times], names, styles):
    plt.semilogy(ts, array-best_val, style, label=name);
plt.ylabel("Objective residual");
plt.xlabel("Time (s)");
plt.ylim(1e-8, 1);
# plt.xlim(0, 100);
plt.legend();
plt.show()

In [ ]:
names = ["NLGCG", "Adaptive Refinement"]
styles = ["-", "--"]
cs = ["blue", "green"]
plt.figure(figsize=(9,5))
for array, name, style, c in zip([supports, actives], names, styles, cs):
    plt.plot(np.arange(len(array)), array, style, label=name, c=c);
plt.ylabel("Number of tunable coefficients");
plt.xlabel("Iterations");
# plt.ylim(1e-8, 1);
# plt.xlim(0, 100);
plt.legend();
plt.show()

# PDEs

## Generate Data and Define Functions

In [ ]:
logging.getLogger().setLevel(logging.INFO) # Supress logging

In [ ]:
sigma_space = np.array([0,1])
omega_space = np.array([[-1,1], [-1,1]])
Omega = np.vstack((sigma_space, omega_space))
d = omega_space.shape[0]
variance_exponent = 2.01
alpha = 1e-1
lambda_ = 1000
observation_resolution = 20

@jax.jit
def true_function(x: np.ndarray):
    return jnp.tanh(4*(0.3-jnp.linalg.norm(x-np.array([0.300000000000000001,0.3000000000000001]))))+jnp.tanh(12*(0.15-jnp.linalg.norm(x+np.array([0.300000000000001,0.3000000000000001]))))+2
grad_true_function = jax.jit(jax.grad(true_function))
hess_true_function = jax.jit(jax.hessian(true_function))
true_function = jax.vmap(true_function)
grad_true_function = jax.vmap(grad_true_function)
hess_true_function = jax.vmap(hess_true_function)

int_function = true_function
grad_int_function = grad_true_function
hess_int_function = hess_true_function
laplacian_int_function = lambda omega: jnp.trace(hess_int_function(omega),axis1=1, axis2=2)
bound_function = true_function

In [ ]:
max_radius = (Omega[0][1]-Omega[0][0])/100

In [ ]:
observations_raw = (np.array(np.meshgrid(
                    *(
                        np.linspace(bound[0], bound[1], observation_resolution, endpoint=True)
                        for bound in omega_space
                    ))
            ).reshape(len(omega_space), -1).T)
int_observations = np.array([obs for obs in observations_raw if all(obs!=omega_space[0][0]) and all(obs!=omega_space[0][1])])
bound_observations = np.array([obs for obs in observations_raw if any(obs==omega_space[0][0]) or any(obs==omega_space[0][1])])

In [ ]:
int_target = int_function(int_observations)**3-laplacian_int_function(int_observations)
bound_target = np.sqrt(lambda_) * bound_function(bound_observations)
target = np.hstack((int_target, bound_target))
observations = np.vstack((int_observations, bound_observations))

constant_dim = len(target)
kernel_dim = len(target)+len(int_target)

In [ ]:
@jax.jit
def adjusted_sigmoid(x):
    return 2*jnp.exp(2*x)/(1+jnp.exp(2*x))-1

@jax.jit
def raw_kernel(omega: np.ndarray, x: np.ndarray):
    # Both inputs 1D
    sigma = omega[0]
    x_omega = omega[1:]
    variance_factor = adjusted_sigmoid(sigma**variance_exponent)
    inner = -(jnp.sum((x_omega - x)**2))/(2*sigma**2)
    outer = (jnp.exp(inner)*variance_factor)/(np.sqrt(2*np.pi)**d)
    return outer

hess_raw_kernel = jax.jit(jax.hessian(raw_kernel, argnums=1))
raw_kernel = jax.vmap(raw_kernel, (None, 0), 0)
hess_raw_kernel = jax.vmap(hess_raw_kernel, (None, 0), 0)

@jax.jit
def singleton_kernel(omega: np.ndarray):
    # function, laplacian
    return jnp.hstack([raw_kernel(omega, observations),jnp.trace(hess_raw_kernel(omega, int_observations),axis1=1, axis2=2)])

grad_kernel = jax.jit(jax.jacobian(singleton_kernel))
hess_kernel = jax.jit(jax.hessian(singleton_kernel))
_ = grad_kernel(np.ones(len(Omega)))
_ = hess_kernel(np.ones(len(Omega)))

kernel = jax.vmap(singleton_kernel)
grad_kernel = jax.vmap(grad_kernel)
hess_kernel = jax.vmap(hess_kernel)

In [ ]:
g = jax.jit(lambda w: alpha * jnp.linalg.norm(w, ord=1))
grad_g = jax.jit(jax.grad(g))
_ = grad_g(jnp.ones(1))

@jax.jit
def r(function_and_laplacian: np.ndarray) -> np.ndarray:
    u = function_and_laplacian[:len(target)] # shape (len(target),)
    laplacian = function_and_laplacian[len(target):] # shape (len(int_target),)
    to_return = jnp.hstack((u[:len(int_target)]**3 - laplacian, np.sqrt(lambda_) * u[len(int_target):]))
    return to_return

f = jax.jit(lambda y: 0.5 * jnp.sum((r(y)-target)**2)*4/len(target))
grad_f = jax.jit(jax.grad(f))
hess_f = jax.jit(jax.hessian(f))
_ = grad_f(jnp.ones(2*len(int_target)+len(bound_target)))
_ = hess_f(jnp.ones(2*len(int_target)+len(bound_target)))

j = lambda u, c: f(u.duality_pairing(kernel, kernel_dim)+np.hstack((c*np.ones(constant_dim), np.zeros(kernel_dim-constant_dim)))) + g(u.coefficients)

In [ ]:
# def p(u: Measure, c: float) -> Callable:
#     inner = -grad_f(u.duality_pairing(kernel, kernel_dim)+np.hstack((c*np.ones(constant_dim), np.zeros(kernel_dim-constant_dim))))
#     return lambda omega: kernel(omega) @ inner

# def grad_p(u: Measure, c: float) -> Callable:
#     inner = -grad_f(u.duality_pairing(kernel, kernel_dim)+np.hstack((c*np.ones(constant_dim), np.zeros(kernel_dim-constant_dim))))
#     return lambda omega: np.tensordot(grad_kernel(omega), inner, axes=([1,0]))

# def hess_p(u: Measure, c: float) -> Callable:
#     inner = -grad_f(u.duality_pairing(kernel, kernel_dim)+np.hstack((c*np.ones(constant_dim), np.zeros(kernel_dim-constant_dim))))
#     return lambda omega: np.tensordot(hess_kernel(omega), inner, axes=([1,0]))

In [ ]:
@jax.jit
def p_raw(parameters: np.ndarray, c: float, omega: np.ndarray):
    coefficients = parameters[:,0]
    support = parameters[:,1:]
    Ku = jnp.tensordot(kernel(support), coefficients, axes=([0], [0]))
    constant_term = jnp.hstack((c*jnp.ones(constant_dim), jnp.zeros(kernel_dim-constant_dim)))
    return singleton_kernel(omega) @ -grad_f(Ku+constant_term)

grad_p_raw = jax.jit(jax.grad(p_raw, argnums=2))
hess_p_raw = jax.jit(jax.hessian(p_raw, argnums=2))

p_raw = jax.jit(jax.vmap(p_raw, in_axes=(None, None, 0), out_axes=0))
grad_p_raw = jax.jit(jax.vmap(grad_p_raw, in_axes=(None, None, 0), out_axes=0))
hess_p_raw = jax.jit(jax.vmap(hess_p_raw, in_axes=(None, None, 0), out_axes=0))

p = lambda u, c: lambda omega: p_raw(u.to_matrix(len(Omega)), c, omega)
grad_p = lambda u, c: lambda omega: grad_p_raw(u.to_matrix(len(Omega)), c, omega)
hess_p = lambda u, c: lambda omega: hess_p_raw(u.to_matrix(len(Omega)), c, omega)

In [ ]:
# Parameterized versions of f and j
@jax.jit
def f_N(raw_input: np.ndarray) -> float:
    constant = raw_input[-1]
    input = raw_input[:-1].reshape(-1, d+2)
    weights = input[:,0]
    omega = input[:,1:]
    return f(kernel(omega).T@weights+constant*jnp.hstack((jnp.ones(constant_dim), jnp.zeros(kernel_dim-constant_dim))))

@jax.jit
def j_N(raw_input: np.ndarray) -> float:
    input = raw_input[:-1].reshape(-1, d+2)
    weights = input[:,0]
    return f_N(raw_input) + g(weights)

grad_f_N = jax.jit(jax.grad(f_N))
hess_f_N = jax.jit(jax.hessian(f_N))
grad_j_N = jax.jit(jax.grad(j_N))
hess_j_N = jax.jit(jax.hessian(j_N))

In [ ]:
# Define functions where the derivatives are taken wrt regularized (weights) and non-regularized(support + constant) parameters

@jax.jit
def f_N_(weights: np.ndarray, support_constant: np.ndarray) -> float:
    constant = support_constant[-1]
    omega = support_constant[:-1].reshape(-1, d+1)
    return f(kernel(omega).T@weights+constant*jnp.ones(target.shape))

@jax.jit
def j_N_(weights: np.ndarray, support_constant: np.ndarray) -> float:
    return f_N_(weights, support_constant) + g(weights)

grad_f_N_reg = jax.jit(jax.grad(f_N_, argnums=0))
hess_f_N_reg = jax.jit(jax.hessian(f_N_, argnums=0))
grad_j_N_reg = jax.jit(jax.grad(j_N_, argnums=0))
hess_j_N_reg = jax.jit(jax.hessian(j_N_, argnums=0))

grad_f_N_non_reg = jax.jit(jax.grad(f_N_, argnums=1))
hess_f_N_non_reg = jax.jit(jax.hessian(f_N_, argnums=1))
grad_j_N_non_reg = jax.jit(jax.grad(j_N_, argnums=1))
hess_j_N_non_reg = jax.jit(jax.hessian(j_N_, argnums=1))

## Experiments

### NLGCG

In [ ]:
exp = NLGCG(target=target, 
           kernel=kernel, 
           g=g,
           f=f,
           f_N=f_N,
           grad_f=grad_f,
           hess_f=hess_f,
           grad_f_N=grad_f_N,
           hess_f_N=hess_f_N,
           grad_f_N_non_reg=grad_f_N_non_reg,
           j=j,
           j_N=j_N,
           j_N_=j_N_,
           p=p,
           grad_p=grad_p,
           hess_p=hess_p,
           grad_j_N=grad_j_N,
           hess_j_N=hess_j_N,
           alpha=alpha,
           Omega=Omega,
           global_search_resolution=20,
           dual_variable_goodness=0.3,
           constant_dim=constant_dim,
           kernel_dim=kernel_dim,
           newton_tolerance=1e-1
           )

In [ ]:
u = pickle.load(open("u1.pkl", "rb")) # 1e-2
c = pickle.load(open("c1.pkl", "rb"))

In [ ]:
u, c, times, supports, inner_loop, lgcg_lazy, lgcg_total, objective_values, dropped_tot, epsilons = exp.solve(tol=5e-14, max_radius=0.01)

In [ ]:
# pickle.dump(u, open("u1.pkl", "wb"))
# pickle.dump(c, open("c1.pkl", "wb"))

In [ ]:
print(u)

In [ ]:
c

In [ ]:
full_parameters = np.hstack((u.to_matrix().flatten(), c))

In [ ]:
grd = exp.grad_j_N(full_parameters)
np.linalg.norm(grd)

In [ ]:
hss = exp.hess_j_N(full_parameters)
np.linalg.eigvals(hss)

### Particle Gradient Descent

In [ ]:
exp = ParticleDescent(
        m=5,
        j=j,
        p=p,
        grad_p=grad_p,
        Omega=Omega,
        a_parameter=0.01,
        b_parameter=0.001,
        kernel=kernel,
        constant_dim=len(target),
        kernel_dim=len(target),
        alpha=alpha,
        target=target,
        g=g,
        f=f,
        grad_f=grad_f,
        hess_f=hess_f,
        ssn_steps=100,
    )

In [ ]:
u, c, objective_values = exp.solve(max_iters=int(1e6))

### Full

In [ ]:
exp_nlgcg = NLGCG(target=target, 
           kernel=kernel, 
           g=g,
           f=f,
           f_N=f_N,
           grad_f=grad_f,
           hess_f=hess_f,
           grad_f_N=grad_f_N,
           hess_f_N=hess_f_N,
           grad_f_N_non_reg=grad_f_N_non_reg,
           j=j,
           j_N=j_N,
           j_N_=j_N_,
           p=p,
           grad_p=grad_p,
           hess_p=hess_p,
           grad_j_N=grad_j_N,
           hess_j_N=hess_j_N,
           alpha=alpha,
           Omega=Omega,
           global_search_resolution=20,
           dual_variable_goodness=0.3,
           constant_dim=constant_dim,
           kernel_dim=kernel_dim,
           newton_tolerance=1e-1
           )

In [ ]:
exp_particle = ParticleDescent(
        m=5,
        j=j,
        p=p,
        grad_p=grad_p,
        Omega=Omega,
        a_parameter=0.001,
        b_parameter=0.001/2,
        kernel=kernel,
        constant_dim=constant_dim,
        kernel_dim=kernel_dim,
        alpha=alpha,
        target=target,
        g=g,
        f=f,
        grad_f=grad_f,
        hess_f=hess_f,
        ssn_steps=100,
    )

In [ ]:
exp_adaptive = AdaptiveRefinement(
        observations=observations,
        variance_exponent=variance_exponent,
        j=j,
        p=p,
        grad_p=grad_p,
        hess_p=hess_p,
        Omega=Omega,
        kernel=kernel,
        constant_dim=constant_dim,
        kernel_dim=kernel_dim,
        alpha=alpha,
        target=target,
        g=g,
        f=f,
        grad_f=grad_f,
        hess_f=hess_f,
        ssn_steps=1000,
    )

In [ ]:
def adapt_time(times, residuals, frame=100, resolution=1):
    to_return = []
    last_pos = 0
    last_res = residuals[0]
    for t in range(frame):
        for i, (res, tim) in enumerate(zip(residuals[last_pos:], times[last_pos:])):
            if tim > t+resolution:
                to_return.append(last_res)
                if tim  - t - resolution < resolution:
                    last_pos += i
                break
            else:
                last_res = res
    to_return.append(last_res)
    return to_return

def bring_to_same_length(arrays, mode):
    max_length = max(len(arr) for arr in arrays)
    new_arrays = []
    for arr in arrays:
        if len(arr) < max_length:
            if mode=="support":
                last_val = arr[-1]
            elif mode=="residual":
                last_val = 0
            arr = arr + [last_val] * (max_length - len(arr))
        new_arrays.append(arr)
    return new_arrays

In [ ]:
optimum = 2.19753851927879e-1
logging.getLogger().setLevel(logging.CRITICAL) # Supress logging

In [ ]:
# deterministic NLGCG
u, c, times_det_nlgcg, supports_det_nlgcg, inner_loop, lgcg_lazy, lgcg_total, objective_values_det_nlgcg, dropped_tot, epsilons = exp_nlgcg.solve(tol=5e-14, max_radius=max_radius, temperature=0.1, mode="deterministic")
residuals_det_nlgcg = adapt_time(times_det_nlgcg, [obj - optimum for obj in objective_values_det_nlgcg], frame=1000, resolution=1)

In [ ]:
# Adaptive refinement
cells_dict, vertices_dict, vertices, u, objective_values_adaptive, times_adaptive, actives, supports_adaptive = exp_adaptive.solve(max_iters=200)
residuals_adaptive = adapt_time(times_adaptive, [obj - optimum for obj in objective_values_adaptive], frame=1000, resolution=1)

In [ ]:
# NLGCG stochastic
nlgcg_residuals = []
nlgcg_supports = []
for i in range(10):
    print(i)
    u, c, times_nlgcg, supports_nlgcg, inner_loop, lgcg_lazy, lgcg_total, objective_values_nlgcg, dropped_tot, epsilons = exp_nlgcg.solve(tol=5e-14, max_radius=max_radius, temperature=0.1)
    local_residuals = adapt_time(times_nlgcg, [obj - optimum for obj in objective_values_nlgcg], frame=1000, resolution=1)
    nlgcg_residuals.append(local_residuals)
    nlgcg_supports.append(supports_nlgcg)
nlgcg_residuals_mean = np.mean(bring_to_same_length(nlgcg_residuals, mode="residual"), axis=0)
nlgcg_residuals_std = np.std(bring_to_same_length(nlgcg_residuals, mode="residual"), axis=0)
nlgcg_supports_mean = np.mean(bring_to_same_length(nlgcg_supports, mode="support"), axis=0)
nlgcg_supports_std = np.std(bring_to_same_length(nlgcg_supports, mode="support"), axis=0)

In [ ]:
# Particle Descent Stochastic
particle_residuals = []
particle_supports = []
for i in range(10):
    print(i)
    u, c, objective_values_particle, supports_particle, times_particle, success = exp_particle.solve(max_iters=int(1e6))
    local_residuals = adapt_time(times_particle, [obj - optimum for obj in objective_values_particle], frame=1000, resolution=1)
    particle_residuals.append(local_residuals)
    particle_supports.append(supports_particle)
particle_residuals_filtered = []
converged_frac = 0
for i in range(len(particle_residuals)):
    if particle_residuals[i][-1] < 1e-6:
        particle_residuals_filtered.append(particle_residuals[i])
        converged_frac += 0.1
particle_residuals_mean = np.mean(bring_to_same_length(particle_residuals_filtered, mode="residual"), axis=0)
particle_residuals_std = np.std(bring_to_same_length(particle_residuals_filtered, mode="residual"), axis=0)
particle_supports_mean = np.mean(bring_to_same_length(particle_supports, mode="support"), axis=0)
particle_supports_std = np.std(bring_to_same_length(particle_supports, mode="support"), axis=0)
print(converged_frac)

In [ ]:
# Plot residuals
fig, ax = plt.subplots(figsize=(5,5))
names = ["NLGCG", "Adaptive Refinement", "SNLGCG", "Particle Descent"]
styles = ["-", "-.", "--", ":"]
for array, name, style in zip([residuals_det_nlgcg, residuals_adaptive, nlgcg_residuals_mean, particle_residuals_mean], names, styles):
    ax.semilogy(np.arange(len(array)), array, style, label=name);
ax.fill(np.hstack((np.arange(len(particle_residuals_mean)), np.arange(len(particle_residuals_mean))[::-1])), np.hstack((np.array(particle_residuals_mean)-np.array(particle_residuals_std), np.array(particle_residuals_mean)[::-1]+np.array(particle_residuals_std)[::-1])), 'red', alpha=0.3);
ax.fill(np.hstack((np.arange(len(nlgcg_residuals_mean)), np.arange(len(nlgcg_residuals_mean))[::-1])), np.hstack((np.array(nlgcg_residuals_mean)-np.array(nlgcg_residuals_std), np.array(nlgcg_residuals_mean)[::-1]+np.array(nlgcg_residuals_std)[::-1])), 'green', alpha=0.3);
plt.ylabel("Objective residual");
plt.xlabel("Time (s)");
plt.ylim(1e-12, 100);
# plt.xlim(0, 100);
ax.legend();

In [ ]:
# Plot supports
fig, ax = plt.subplots(figsize=(5,5))
names = ["NLGCG", "Adaptive Refinement", "SNLGCG", "Particle Descent"]
styles = ["-", "-.", "--", ":"]
for array, name, style in zip([supports_det_nlgcg, supports_adaptive, nlgcg_supports_mean, particle_supports_mean], names, styles):
    ax.semilogx(np.arange(len(array)), array, style, label=name);
ax.fill(np.hstack((np.arange(len(particle_supports_mean)), np.arange(len(particle_supports_mean))[::-1])), np.hstack((np.array(particle_supports_mean)-np.array(particle_supports_std), np.array(particle_supports_mean)[::-1]+np.array(particle_supports_std)[::-1])), 'red', alpha=0.3);
ax.fill(np.hstack((np.arange(len(nlgcg_supports_mean)), np.arange(len(nlgcg_supports_mean))[::-1])), np.hstack((np.array(nlgcg_supports_mean)-np.array(nlgcg_supports_std), np.array(nlgcg_supports_mean)[::-1]+np.array(nlgcg_supports_std)[::-1])), 'green', alpha=0.3);
plt.ylabel("Support points");
plt.xlabel("Iterations");
# plt.ylim(1e-12, 100);
# plt.xlim(0, 100);
ax.legend();

In [ ]:
# Plot number of coefficients to potimize
fig, ax = plt.subplots(figsize=(5,5))
names = ["NLGCG", "Adaptive Refinement", "SNLGCG"]
styles = ["-", "-.", "--"]
for array, name, style in zip([supports_det_nlgcg, actives, nlgcg_supports_mean], names, styles):
    ax.semilogx(np.arange(len(array)), array, style, label=name);
ax.fill(np.hstack((np.arange(len(nlgcg_supports_mean)), np.arange(len(nlgcg_supports_mean))[::-1])), np.hstack((np.array(nlgcg_supports_mean)-np.array(nlgcg_supports_std), np.array(nlgcg_supports_mean)[::-1]+np.array(nlgcg_supports_std)[::-1])), 'green', alpha=0.3);
plt.ylabel("Number of coefficients to optimize");
plt.xlabel("Iterations");
# plt.ylim(1e-12, 100);
# plt.xlim(0, 100);
ax.legend();

## Plots

In [ ]:
parameters, u_ks, radii = exp.local_merging_update_radii(u, c)
radii

In [ ]:
# Projection of the dual variable on an axis
p_u = p(u, c)
resolution = 100
a_sigma = np.linspace(0,1,resolution, endpoint=False)
index = -26
points = np.array([u.support[index]]*resolution)
points[:,0]=a_sigma
vals = np.abs(p_u(points))

plt.plot(a_sigma, vals);
plt.plot([u.support[index][0]], np.abs(p_u(np.array([u.support[index]]))), "o", c="black", markersize=5);

In [ ]:
# 2D problem with a true function and predicted sources
resolution = 100
a_1 = np.linspace(omega_space[0][0],omega_space[0][1],resolution, endpoint=False)
a_2 = np.linspace(omega_space[1][0],omega_space[1][1],resolution, endpoint=False)
x, y = np.meshgrid(a_1,a_2)
points = np.array(list(zip(x.flatten(), y.flatten())))
vals = int_function(points).reshape((resolution,resolution))

@jax.jit
def plot_kernel(omega: np.ndarray):
    return raw_kernel(omega, points)
plot_kernel = jax.vmap(plot_kernel)

pred_vals = u.duality_pairing(plot_kernel).reshape((resolution,resolution)) +c*np.ones((resolution, resolution))

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15,4))
contour1 = ax1.contourf(x, y, vals, levels=100)
fig.colorbar(contour1, ax=ax1)
contour2 = ax2.contourf(x, y, pred_vals, levels=100)
fig.colorbar(contour2, ax=ax2)
contour3 = ax3.contourf(x, y, np.abs(pred_vals-vals), levels=100)
fig.colorbar(contour3, ax=ax3)
for i, x in enumerate(u.support):
    if u.coefficients[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(u.coefficients[i])
    ax2.add_patch(plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5))
    ax3.add_patch(plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5))
ax1.set_xlim(omega_space[0][0],omega_space[0][1])
ax1.set_ylim(omega_space[1][0],omega_space[1][1])
ax2.set_xlim(omega_space[0][0],omega_space[0][1])
ax2.set_ylim(omega_space[1][0],omega_space[1][1])
ax3.set_xlim(omega_space[0][0],omega_space[0][1])
ax3.set_ylim(omega_space[1][0],omega_space[1][1])

logging.info(f"L2: {np.linalg.norm(pred_vals-vals)/len(vals)}, Linf: {np.max(np.abs(pred_vals-vals))}")

In [ ]:
residuals = np.array(objective_values) - np.min(objective_values)
plt.figure(figsize=(11.25,5))
plt.semilogy(np.array(range(len(residuals)-1)), residuals[:-1], linestyle="-.", color="green");
plt.ylim(1e-17, 10000);
plt.ylabel("Objective residual");
plt.xlabel("Total iterations");
plt.show()

In [ ]:
s_times = np.array([2.38418579e-07, 1.40449047e+00, 1.05085146e+01, 1.98205879e+01,
       2.80613465e+01, 3.62870891e+01, 4.02740636e+01, 4.83748887e+01,
       5.20582135e+01, 5.95958805e+01, 6.71428375e+01, 7.02771153e+01,
       7.36518626e+01, 7.66863663e+01, 8.02852213e+01, 8.20774333e+01,
       8.52600460e+01, 8.69454968e+01, 8.87883384e+01, 9.03524165e+01,
       9.36496050e+01])

s_res = np.array([9.52366198e+02, 9.52275334e+02, 1.20388408e+02, 3.75742355e+01,
       1.03593709e+01, 7.63087884e+00, 3.57120016e+00, 1.61612115e+00,
       1.16712560e-01, 4.66542758e-02, 1.91823265e-04, 2.69549673e-05,
       2.66407815e-06, 5.12761687e-07, 5.31816795e-08, 8.71679617e-09,
       9.06197783e-10, 1.59531055e-10, 3.84545729e-11, 7.47490958e-12,
       0.00000000e+00])

l_times = np.array([2.38418579e-07, 1.46344781e+00, 8.11061549e+00, 2.57419589e+01,
       3.98512027e+01, 4.99800742e+01, 6.30679688e+01, 6.57149813e+01,
       7.46773405e+01, 8.79460781e+01, 9.59659581e+01, 1.04081795e+02,
       1.11707988e+02, 1.19354894e+02, 1.27378870e+02, 1.35241861e+02,
       1.42994247e+02, 1.50890936e+02, 1.58865531e+02, 1.66695525e+02,
       1.74381387e+02, 1.82145741e+02])

l_res = np.array([9.52366198e+02, 9.52275334e+02, 4.41972641e+01, 3.54211183e+01,
       3.20555932e+00, 1.95996832e+00, 9.60235862e-01, 6.82964577e-02,
       2.56595990e-02, 2.94252555e-04, 6.76416836e-05, 1.29407016e-05,
       2.79499673e-06, 7.47630082e-07, 1.49852013e-07, 3.32056800e-08,
       6.84104862e-09, 1.51797508e-09, 1.61406888e-10, 3.41913164e-11,
       9.15179044e-12, 3.78008735e-12])

f_times = np.array([2.38418579e-07, 1.03882067e+01, 2.28253095e+01, 3.45720618e+01,
       4.63346055e+01, 5.75706902e+01, 6.42947633e+01, 7.80636165e+01,
       9.25107338e+01, 1.05564734e+02, 1.19229394e+02, 1.27402204e+02,
       1.37276371e+02, 1.49799072e+02, 1.57690953e+02, 1.65466475e+02,
       1.73376557e+02, 1.81083387e+02, 1.88917793e+02, 1.96844106e+02,
       2.04801170e+02, 2.12726861e+02, 2.20301038e+02, 2.28052228e+02,
       2.35463697e+02])

f_res = np.array([9.52366198e+02, 9.52275334e+02, 1.77621316e+02, 4.40709130e+01,
       3.74052976e+01, 3.59800630e+01, 3.54530562e+01, 1.20709990e+01,
       3.42101480e+00, 1.42317610e+00, 8.29428755e-01, 2.32684019e-02,
       3.42311798e-03, 8.63112579e-05, 1.50197302e-05, 3.03893884e-06,
       4.90991397e-07, 6.19446325e-08, 7.72831754e-09, 1.53960400e-09,
       2.56221711e-10, 5.43991518e-11, 1.31024080e-11, 1.53477231e-12,
       1.50635060e-12])

In [ ]:
names = ["GCG", "LGCG", "LSGCG"]
styles = ["-", "--", "-.", ":"]
plt.figure(figsize=(9,5))
for domain, array, name, style in zip([f_times, l_times, s_times], [f_res, l_res, s_res], names, styles):
    plt.semilogy(domain, array, style, label=name);
plt.ylabel("Objective residual");
plt.xlabel("Time (s)");
plt.ylim(1e-12, 1100);
# plt.xlim(0, 60);
plt.legend();
plt.show()

# Function Approximation (Sigmoid Shallow NN)

## Generate Data and Define Functions

In [ ]:
alpha = 1e-6

In [ ]:
# Function approximation
observation_size = 100
observation_space = np.array([[0,1], [0,1]])
Omega = np.array([[0,1], [0,1], [0,1]])
true_function = lambda x: np.sin(10*(x[0]**2+x[1]**2))

np.random.seed(49)
columns = []
for bounds in observation_space:
    columns.append(
        np.random.sample((observation_size, 1)) * (bounds[1] - bounds[0]) + bounds[0]
    )
observations = np.concatenate(columns, axis=1)
observations_bias = np.append(np.ones((observations.shape[0],1)),observations,axis=1)
target = np.array([true_function(x) for x in observations])

In [ ]:
activation = lambda omega: 1/(1+np.exp(omega)) # sigmoid
def kappa(x):
    # input is a 2D array of shape (number of points, dimension of observation space +1)
    if len(x.shape) == 1:
        x = x.reshape(1, -1)
    linear_operation = x@(observations_bias.T)
    activated = activation(linear_operation)
    return activated # shape (len(x), len(observations))

In [ ]:
def activation_grad(omega: np.ndarray) -> np.ndarray:
    sigmoid = activation(omega)
    return sigmoid * (1 - sigmoid)  # Derivative of the sigmoid function

def grad_kappa(x):
    # input is a 2D array of shape (number of points, dimension of observation space +1)
    if len(x.shape) == 1:
        x = x.reshape(1, -1)
    linear_operation = x@(observations_bias.T)
    activated = activation_grad(linear_operation) # shape=(len(x), len(observations))
    inner_derivative = np.repeat(np.expand_dims(observations_bias,axis=0),len(x),axis=0) # shape=(len(x), len(observations), Omega.shape[0])
    chain_rule = np.multiply(activated.reshape((len(x),len(observations),1)), inner_derivative)
    return chain_rule # shape (len(x), len(observations), Omega.shape[0])

In [ ]:
def activation_hess(omega: np.ndarray) -> np.ndarray:
    sigmoid = activation(omega)
    return sigmoid * (1 - sigmoid)*(1-2*sigmoid)  # Derivative of the sigmoid function

def hess_kappa(x):
    # Input is 2D array of shape (number of points, dimension of observation space +1)
    if len(x.shape) == 1:
        x = x.reshape(1, -1)
    linear_operation = x@(observations_bias.T)
    activated = activation_hess(linear_operation)
    inner_second_derivative_raw = np.array([np.outer(vect, vect) for vect in observations_bias]) # shape (len(observations),Omega.shape[0],Omega.shape[0])
    inner_second_derivative = np.repeat(np.expand_dims(inner_second_derivative_raw,axis=0),len(x),axis=0) # shape (len(x),len(observations),Omega.shape[0],Omega.shape[0])
    chain_rule = np.multiply(activated.reshape((len(x),len(observations),1,1)), inner_second_derivative)
    return chain_rule # shape (len(x), len(observations), Omega.shape[0], Omega.shape[0])

In [ ]:
g = lambda u: alpha * np.linalg.norm(u, ord=1)
f = lambda u: 0.5 * np.linalg.norm(u.duality_pairing(kappa) - target) ** 2

In [ ]:
def p_raw(u):
    Ku = u.duality_pairing(kappa)
    inner = Ku-target
    return lambda x: -kappa(x) @ inner

p = lambda u: p_raw(u)

In [ ]:
def grad_P_raw(u):
    p_u = p_raw(u)
    inner = target-u.duality_pairing(kappa)
    return lambda x: np.sign(p_u(x)).reshape(-1,1)*np.tensordot(grad_kappa(x), inner, axes=([1,0]))

grad_P = lambda u: grad_P_raw(u)

In [ ]:
def grad_P_raw_sphere(u):
    p_u = p_raw(u)
    inner = target-u.duality_pairing(kappa)
    def grad_P_x(x):
        unprojected = np.sign(p_u(x)).reshape(-1,1)*np.tensordot(grad_kappa(x), inner, axes=([1,0]))
        to_return = np.zeros(unprojected.shape)
        for i, (grad ,position) in enumerate(zip(unprojected, x)):
            # Project onto the ball
            projection = np.eye(len(position)) - np.outer(position, position)
            to_return[i] = projection@grad
        return to_return
    return grad_P_x

grad_P_sphere = lambda u: grad_P_raw_sphere(u)

In [ ]:
def hess_P_raw(u):
    p_u = p_raw(u)
    inner = target-u.duality_pairing(kappa)
    return lambda x: np.sign(p_u(x)).reshape(-1,1,1)*np.tensordot(hess_kappa(x),inner,axes=([1,0]))

hess_P = lambda u: hess_P_raw(u)

In [ ]:
def hess_P_raw_sphere(u):
    p_u = p_raw(u)
    grad_u = grad_P(u)
    inner = target-u.duality_pairing(kappa)
    def hess_P_x(x):
        grads = grad_u(x)
        unprojected = np.sign(p_u(x)).reshape(-1,1,1)*np.tensordot(hess_kappa(x),inner,axes=([1,0]))
        to_return = np.zeros(unprojected.shape)
        for i, (grad, hess, position) in enumerate(zip(grads, unprojected, x)):
            # P_x(hess-position.T*grad*I)
            projection = np.eye(len(position)) - np.outer(position, position)
            identity_factor = position@grad
            to_return[i] = projection@(hess-identity_factor*np.eye(len(position)))@projection
        return to_return
    return hess_P_x

hess_P_sphere = lambda u: hess_P_raw_sphere(u)

In [ ]:
def grad_j(positions, coefs):
    K_matrix = kappa(positions)
    grad_F = (K_matrix.T@coefs).flatten() - target
    nabla_x = coefs.reshape(-1,1)*np.tensordot(grad_kappa(positions), grad_F, axes=([1,0]))
    nabla_u = np.dot(K_matrix, grad_F) + alpha * np.sign(coefs)
    return np.append(nabla_x.flatten(), nabla_u, axis=0).flatten()

In [ ]:
def grad_j_sphere(positions, coefs):
    K_matrix = kappa(positions)
    grad_F = (K_matrix.T@coefs).flatten() - target
    nabla_x = coefs.reshape(-1,1)*np.tensordot(grad_kappa(positions), grad_F, axes=([1,0]))
    for i, (grad ,position) in enumerate(zip(nabla_x, positions)):
        # Project onto the ball
        nabla_x[i] = (np.eye(len(position))-np.outer(position, position))@grad
    nabla_u = np.dot(K_matrix, grad_F) + alpha * np.sign(coefs)
    return np.append(nabla_x.flatten(), nabla_u, axis=0).flatten()

In [ ]:
def hess_j(positions, coefs):
    kappa_values = kappa(positions)
    grad_kappa_values = grad_kappa(positions)
    hess_kappa_values = hess_kappa(positions)
    matrix_dimension = len(positions)*Omega.shape[0] + len(coefs)
    hesse_matrix = np.zeros((matrix_dimension, matrix_dimension))
    step = Omega.shape[0]
    coefs_delay = step*len(positions)
    inner = (kappa_values.T@coefs).flatten() - target
    for i in range(len(positions)):
        # nabla_{x_i,x_j}
        for j in range(len(positions)):
            if j<i:
                continue
            block = coefs[i]*coefs[j]*np.matmul(grad_kappa_values[i].T, grad_kappa_values[j])
            if i==j:
                block += coefs[i]*np.tensordot(hess_kappa_values[i],inner,axes=([0,0]))
            hesse_matrix[i*step:(i+1)*step, j*step:(j+1)*step] = block
            hesse_matrix[j*step:(j+1)*step, i*step:(i+1)*step] = block.T
        # nabla_{x_i,u_j}
        for j in range(len(coefs)):
            block = coefs[i]*np.matmul(grad_kappa_values[i].T, kappa_values[j])
            if i == j:
                block += np.matmul(grad_kappa_values[i].T, inner)
            hesse_matrix[i*step:(i+1)*step, coefs_delay+j] = block
            hesse_matrix[coefs_delay+j, i*step:(i+1)*step] = block.T
    for i in range(len(coefs)):
        # nabla_{u_i,u_j}
        for j in range(len(coefs)):
            if j<i:
                continue
            block = np.dot(kappa_values[i], kappa_values[j])
            hesse_matrix[coefs_delay+i,coefs_delay+j] = block
            hesse_matrix[coefs_delay+j,coefs_delay+i] = block
    return hesse_matrix

In [ ]:
def hess_j_sphere(positions, coefs):
    hessian = hess_j(positions, coefs)
    gradient = grad_j(positions, coefs)
    projection = np.eye(hessian.shape[0])
    inner_matrix = np.eye(hessian.shape[0])
    for i, position in enumerate(positions):
        projection_part = np.eye(len(position))-np.outer(position, position)
        inner_matrix_part = (position@gradient[i*len(position):(i+1)*len(position)])*np.eye(len(position))
        projection[i*len(position):(i+1)*len(position), i*len(position):(i+1)*len(position)] = projection_part
        inner_matrix[i*len(position):(i+1)*len(position), i*len(position):(i+1)*len(position)] = inner_matrix_part
    hesse_system = projection@(hessian-inner_matrix)@projection
    return hesse_system

## Experiments

In [ ]:
exp = NLGCG(target=target, 
           kappa=kappa, 
           g=g, 
           f=f,
           p=p,
           grad_P=grad_P_sphere,
           hess_P=hess_P_sphere,
           grad_j=grad_j_sphere,
           hess_j=hess_j_sphere,
           alpha=alpha,
           Omega=Omega,
           global_search_resolution=100,
           dual_variable_goodness=0.3,
           projection="sphere"
           )

In [ ]:
u, times, supports, inner_loop, lgcg_lazy, lgcg_total, objective_values, dropped_tot, epsilons = exp.nlgcg(tol=1e-12, max_radius=0.1)

## Plots

In [ ]:
a = np.arange(0,1,0.01)
B, D = np.meshgrid(a,a)
X = np.array(list(zip(B.flatten(), D.flatten())))
true_vals = np.array([true_function(x) for x in X]).reshape((100,100))

plt.contourf(B, D, true_vals, levels=100);
plt.colorbar();
plt.xlim(Omega[0][0], Omega[0][1]);
plt.ylim(Omega[1][0], Omega[1][1]);

In [ ]:
def predicted(x):
    # Input is 2D array of shape (number of points, Omega dimension)
    if len(x.shape) == 1:
        x = x.reshape(1, -1)
    x_bias = np.append(np.ones((x.shape[0],1)),x,axis=1)
    activated = activation(x_bias@u.support.T) # shape (len(x), len(observations))
    output = np.dot(activated, u.coefficients) # shape (len(x),)
    return output

a = np.arange(0,1,0.01)
B, D = np.meshgrid(a,a)
x = np.array(list(zip(B.flatten(), D.flatten())))
pred_vals = predicted(x).reshape((100,100))
error = true_vals - pred_vals

print(f"L2 error: {np.linalg.norm(error)/np.sqrt(len(x)):.3E}, Linf error: {np.max(np.abs(error)):.3E}")

plt.contourf(B, D, np.abs(error), levels=100);
plt.colorbar();
for i, x in enumerate(observations):
    plt.plot([x[0]], [x[1]], "P", c="r", markersize=4);